In [ ]:
# ══════════════════════════════════════════════════════════════
# Cell 0: 全局配置 - 所有 API 密钥和模型参数集中管理
# ══════════════════════════════════════════════════════════════

# [迁移改动] 新增 CONFIG 字典，集中管理所有 API 密钥和配置
# 原代码中 API Key 分散在 SciverseClient 创建、DeepSeek 初始化、model 变量等多处硬编码
# 现在统一在此处定义，后续所有引用处改为从 CONFIG 读取

import os

CONFIG = {
    # ── Sciverse 学术搜索引擎 API Token ──
    # 用途：认证 Sciverse meta-search API，用于文献检索
    # 当前值：os.environ.get("SCIVERSE_TOKEN", "")
    # 如果要换：到 https://sciverse.space 申请新的 Token，粘贴替换此值
    # 修改此值时需要同步检查：SCIVERSE_DATA_DIR（数据目录）、sciverse_client 实例化
    "sciverse_token": os.environ.get("SCIVERSE_TOKEN", ""),

    # ── DeepSeek LLM API Key ──
    # 用途：调用大语言模型（用于 Agent 推理、GA 评估、可证伪性检查等）
    # 当前值：os.environ.get("DEEPSEEK_API_KEY", "")
    # 如果要换：到 https://platform.deepseek.com/api_keys 生成新 Key，粘贴替换
    # 如果换提供商（如 OpenAI、智谱）：同时修改 llm_base_url 和 llm_model
    # 修改此值时需要同步检查：llm_base_url、llm_model
    "deepseek_key": os.environ.get("DEEPSEEK_API_KEY", ""),

    # ── LLM API 基础 URL ──
    # 用途：指定 LLM API 的端点地址
    # 当前值：https://api.deepseek.com/v1 （DeepSeek 官方端点）
    # 如果换提供商：
    #   - OpenAI:     "https://api.openai.com/v1"
    #   - 智谱 GLM:   "https://open.bigmodel.cn/api/paas/v4"
    #   - 本地 Ollama: "http://localhost:11434/v1"
    #   - 硅基流动:    "https://api.siliconflow.cn/v1"
    # 修改此值时需要同步检查：deepseek_key、llm_model
    "llm_base_url": "https://api.deepseek.com/v1",

    # ── LLM 模型名称 ──
    # 用途：指定调用的具体模型，影响推理质量和速度
    # 当前值：deepseek-v4-pro（关闭思考模式，适配结构化JSON和工具调用）
    # 如果要换模型：
    #   - deepseek-v4-pro     （DeepSeek V4 高性能模型）
    #   - deepseek-v4-flash   （DeepSeek V4 快速模型）
    #   - gpt-4o              （OpenAI 模型）
    #   - glm-4-flash         （智谱模型）
    #   - qwen/qwen2.5-72b-instruct （硅基流动上的通义千问）
    # ⚠️ 注意：本项目的结构化解析和工具调用统一关闭思考模式
    # 修改此值时需要同步检查：deepseek_key、llm_base_url
    "llm_model": "deepseek-v4-pro",

    # ── 搜索阶段轮次配置 [v3新增] ──
# 双循环阶段化搜索：每个阶段的最大轮次
# 总计 12 轮（3+2+1+3+2+1）
"search_phases": {
    "diverge_1": 3,      # 发散搜索1（强检索+弱检索+关联发散）
    "converge_1": 2,     # 收敛搜索1（聚焦假设+补充证据）
    "falsify": 1,        # 证伪检查（判断方向可行性）
    "diverge_2": 3,      # 拓展发散（变体搜索+作者追踪+跨领域类比）
    "converge_2": 2,     # 收敛整合（整合发现+构建证据链）
    "final_verify": 1,   # 最终验证（输出JSON结果）
},
}

print("✅ CONFIG 加载完成")
for k, v in CONFIG.items():
    if "key" in k.lower() or "token" in k.lower():
        masked = v[:4] + "****" + v[-3:] if len(v) > 7 else "****"
        print(f"   {k}: {masked}")
    else:
        print(f"   {k}: {v}")



'''# ── 搜索阶段轮次配置 [v3新增] ──
# 双循环阶段化搜索：每个阶段的最大轮次
# 总计 12 轮（3+2+1+3+2+1）
"search_phases": {
    "diverge_1": 3,      # 发散搜索1（强检索+弱检索+关联发散）
    "converge_1": 2,     # 收敛搜索1（聚焦假设+补充证据）
    "falsify": 1,        # 证伪检查（判断方向可行性）
    "diverge_2": 3,      # 拓展发散（变体搜索+作者追踪+跨领域类比）
    "converge_2": 2,     # 收敛整合（整合发现+构建证据链）
    "final_verify": 1,   # 最终验证（输出JSON结果）
},
# ── [v3新增] 搜索流程阶段配置 ──
# 双循环阶段化搜索：每个阶段的最大工具调用轮次数
# 总计 12 轮，分 6 个阶段，形成 "发散→收敛→证伪→拓展→整合→验证" 的完整研究闭环
# 调整方法：修改对应阶段的数值即可，例如想给发散搜索更多轮次，将 diverge_1 改为 4
# ⚠️ 注意：总轮次会相应变化，max_tool_rounds 参数可能需要同步调整
"search_phases": {
    "diverge_1": 3,      # 阶段1-发散搜索：广泛搜索相关文献（强检索+弱检索+关联发散）
    "converge_1": 2,     # 阶段2-收敛搜索：聚焦 top 假设，精确验证，补充证据缺口
    "falsify": 1,        # 阶段3-证伪检查：对最有证据的假设调用 falsification_check
    "diverge_2": 3,      # 阶段4-拓展发散：变体搜索+作者追踪+跨领域类比
    "converge_2": 2,     # 阶段5-收敛整合：整合所有发现，构建完整证据链
    "final_verify": 1    # 阶段6-最终验证：输出结构化 JSON 结果
}
# 总计: 3+2+1+3+2+1 = 12 轮'''


In [ ]:
# -*- coding: utf-8 -*-
# Cell 1: 状态初始化

# ══════ 🔒 线性防护：检查前置依赖 ══════

_required = ["CONFIG"]
_missing = [v for v in _required if v not in dir() and v not in globals()]
if _missing:
    raise RuntimeError(f"❌ 请先运行定义以下变量的 Cell: {_missing}")

    # ══════ 🔒 线性防护 - 通过 ══════

    # [迁移改动] state 增加 errors 统计字段，增加 ga_complete 标记

    # 原版只有 total_calls 和 cache_hits，现在增加 errors 用于追踪 API 调用失败次数

state = {
"phase": "research", # "research" = 文献调研阶段, "discovery" = 任务A（GA材料搜索）阶段
"context": {
"gaps": [], # Phase 1 发现的 Research Gap 列表，Phase 2 会读取
"cached_literature": [], # 核心文献缓存
"target_property": "" # 用户指定的性能指标
},
"api_stats": {
"total_calls": 0, # Sciverse API 总调用次数（不含缓存命中）
"cache_hits": 0, # 缓存命中次数
"errors": 0 # [迁移改动] API 调用失败次数
}
}

# 持久化初始状态

import json
with open("agent_state.json", "w", encoding="utf-8") as f:
    json.dump(state, f, ensure_ascii=False, indent=2)

    print("✅ 状态初始化完成，可继续执行后续Cell")
    print(f" phase={state['phase']}, api_stats={state['api_stats']}")

In [ ]:
# -*- coding: utf-8 -*-
# Cell 2: 路径与配置
# ============================================================

import os

SCIVERSE_DATA_DIR = r"D:\AI科研助手专用目录"
try:
    os.makedirs(SCIVERSE_DATA_DIR, exist_ok=True)
except OSError as e:
    # D盘不存在或无权限时，回退到脚本所在目录
    fallback = os.path.join(os.getcwd(), "AI科研助手专用目录")
    print(f"⚠️ 无法创建 {SCIVERSE_DATA_DIR}（{e}），已回退至: {fallback}")
    SCIVERSE_DATA_DIR = fallback
    os.makedirs(SCIVERSE_DATA_DIR, exist_ok=True)

CACHE_FILE_PATH = os.path.join(SCIVERSE_DATA_DIR, "cache.json")
LOG_FILE_PATH = os.path.join(SCIVERSE_DATA_DIR, "log.txt")

# [迁移改动] 不再单独定义 SCIVERSE_API_TOKEN 变量，直接从 CONFIG 读取

# 原版: SCIVERSE_API_TOKEN = "sci_xxx".strip()

SCIVERSE_API_TOKEN = CONFIG["sciverse_token"]  # [迁移改动] 从 CONFIG 集中读取

# ============================================================

import json
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

def check_sciverse_connection(token: str) -> bool:
    """
    检测 Sciverse API 是否接入成功
    发送一个最小化的 meta-search 请求，根据响应判断连通性
    :param token: Bearer API Token
    :return: True=接入成功, False=接入失败
    """
    url = "https://api.sciverse.space/meta-search"
    payload = json.dumps({
        "query": "test",
        "page": 1,
        "page_size": 1,
    }).encode("utf-8")
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    }

    print("=" * 50)
    print("🔍 正在检测 Sciverse API 接入状态...")
    print("=" * 50)

    # 检查Token是否为空或占位符
    if not token or token in ("YOUR_API_TOKEN", "在这里粘贴你的真实Token"):
        print("❌ 接入失败：尚未填写有效的 API Token")
        print("   请在 CONFIG['sciverse_token'] 中填入你的真实Token")
        print("=" * 50)
        return False

    try:
        req = Request(url, data=payload, headers=headers, method="POST")
        with urlopen(req, timeout=15) as resp:
            status = resp.status
            body = json.loads(resp.read().decode("utf-8"))

            has_results = "results" in body or "data" in body or "hits" in body
            if status == 200 and has_results:
                print(f"✅ 接入成功！HTTP {status}")
                print(f"   返回数据键: {list(body.keys())}")
                result_count = len(body.get("results", body.get("data", body.get("hits", []))))
                print(f"   测试查询返回 {result_count} 条结果")
                print("=" * 50)
                return True
            else:
                print(f"⚠️  响应异常：HTTP {status}，但返回结构不符合预期")
                print(f"   返回内容: {json.dumps(body, ensure_ascii=False)[:200]}")
                print("=" * 50)
                return False

    except HTTPError as e:
        error_body = ""
        try:
            error_body = e.read().decode("utf-8")[:200]
        except Exception:
            pass
        if e.code == 401:
            print(f"❌ 接入失败：认证错误 (HTTP 401)")
            print(f"   Token 无效或已过期，请检查后重试")
        elif e.code == 403:
            print(f"❌ 接入失败：权限不足 (HTTP 403)")
            print(f"   当前账号可能未开通 meta-search 权限")
        elif e.code == 429:
            print(f"⚠️  触发限流 (HTTP 429)，但说明Token有效、接入正常")
            print(f"   请稍后再运行检测")
            print("=" * 50)
            return True  # 429说明认证通过了
        else:
            print(f"❌ 接入失败：HTTP {e.code}")
        if error_body:
            print(f"   服务端响应: {error_body}")
        print("=" * 50)
        return False

    except URLError as e:
        print(f"❌ 接入失败：网络错误")
        print(f"   原因: {e.reason}")
        print(f"   请检查网络连接及代理设置")
        print("=" * 50)
        return False

    except Exception as e:
        print(f"❌ 接入失败：未知错误 - {type(e).__name__}: {e}")
        print("=" * 50)
        return False

# 🚀 脚本加载时自动执行检测

# [迁移改动] 传入 CONFIG["sciverse_token"] 而非硬编码变量

_api_connected = check_sciverse_connection(CONFIG["sciverse_token"])

# 持久化状态

with open("agent_state.json", "w", encoding="utf-8") as f:
    json.dump(state, f, ensure_ascii=False, indent=2)


In [ ]:
# -*- coding: utf-8 -*-
# Cell 3: Sciverse API 客户端 (v5.1.1)
# 支持 4 个官方端点：meta-search / agentic-search / content / meta-paper-relations
# 修复：移除 is_content_accessible；缓存 key 区分 endpoint；错误结构化返回

import json
import hashlib
import time
import threading
import os
from datetime import datetime
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

class SciverseClient:
    """
    Sciverse API 多端点调用封装
    - 内置 JSON 文件缓存（key 包含 endpoint + payload MD5）
    - 令牌桶限速器（所有端点共享）
    - 429/5xx 自动指数退避重试
    - 400 错误记录详细请求/响应并返回结构化错误
    - 仅使用 Python 标准库
    """

    BASE_URLS = {
        "meta-search": "https://api.sciverse.space/meta-search",
        "agentic-search": "https://api.sciverse.space/agentic-search",
        "content": "https://api.sciverse.space/content",
        "meta-paper-relations": "https://api.sciverse.space/meta-paper-relations",
        "meta-catalog": "https://api.sciverse.space/meta-catalog",
    }

    CACHE_FILE = "sciverse_cache.json"
    LOG_FILE = "log.txt"
    MAX_REQUESTS_PER_MINUTE = 25
    MAX_RETRIES = 3

    def __init__(self, api_token: str, state: dict = None):
        self.api_token = api_token
        self.state = state
        self.cache = self._load_cache()
        self.cache_lock = threading.Lock()

        self.token_bucket_lock = threading.Lock()
        self.tokens = self.MAX_REQUESTS_PER_MINUTE
        self.last_refill_time = time.monotonic()
        self.refill_rate = self.MAX_REQUESTS_PER_MINUTE / 60.0

    # ==================== 缓存 ====================

    def _load_cache(self) -> dict:
        if os.path.exists(self.CACHE_FILE):
            try:
                with open(self.CACHE_FILE, "r", encoding="utf-8") as f:
                    return json.load(f)
            except (json.JSONDecodeError, IOError):
                return {}
        return {}

    def _save_cache(self):
        try:
            with open(self.CACHE_FILE, "w", encoding="utf-8") as f:
                json.dump(self.cache, f, ensure_ascii=False, indent=2)
        except IOError as e:
            self._write_log("SYSTEM", "CACHE_SAVE_ERROR", False, 0, str(e))

    @staticmethod
    def _make_cache_key(endpoint: str, payload: dict) -> str:
        raw = f"{endpoint}:{json.dumps(payload, sort_keys=True, ensure_ascii=False)}"
        return hashlib.md5(raw.encode("utf-8")).hexdigest()

    # ==================== 限速 ====================

    def _acquire_token(self):
        while True:
            with self.token_bucket_lock:
                now = time.monotonic()
                elapsed = now - self.last_refill_time
                new_tokens = elapsed * self.refill_rate
                self.tokens = min(self.MAX_REQUESTS_PER_MINUTE, self.tokens + new_tokens)
                self.last_refill_time = now

                if self.tokens >= 1.0:
                    self.tokens -= 1.0
                    return

            wait_time = (1.0 - self.tokens) / self.refill_rate
            time.sleep(wait_time)

    # ==================== 日志 ====================

    def _write_log(self, timestamp: str, query: str, cache_hit: bool,
                   duration_ms: float, extra: str = ""):
        cache_str = "HIT" if cache_hit else "MISS"
        line = f"[{timestamp}] | {query} | {cache_str} | {duration_ms:.1f}ms"
        if extra:
            line += f" | {extra}"
        line += "\n"
        try:
            with open(self.LOG_FILE, "a", encoding="utf-8") as f:
                f.write(line)
        except IOError:
            pass

    # ==================== HTTP 请求 ====================

    def _do_request(self, endpoint: str, payload: dict) -> dict:
        """
        通用请求方法。
        成功：返回 API 响应 dict。
        失败：返回 {"error": True, "status_code": ..., "message": ..., "endpoint": ..., "payload": ...}
        """
        data = json.dumps(payload).encode("utf-8")
        headers = {
            "Authorization": f"Bearer {self.api_token}",
            "Content-Type": "application/json",
        }

        url = self.BASE_URLS.get(endpoint, endpoint)
        last_exception = None

        for attempt in range(self.MAX_RETRIES + 1):
            try:
                req = Request(url, data=data, headers=headers, method="POST")
                with urlopen(req, timeout=30) as resp:
                    body = resp.read().decode("utf-8")
                    return json.loads(body)

            except HTTPError as e:
                last_exception = e
                status_code = e.code
                try:
                    err_body = e.read().decode("utf-8")
                except Exception:
                    err_body = ""
                try:
                    err_json = json.loads(err_body) if err_body else {}
                except Exception:
                    err_json = {"raw": err_body}

                # 400 记录详细错误并直接返回结构化错误
                if status_code == 400:
                    return {
                        "error": True,
                        "status_code": 400,
                        "message": f"HTTP 400 Bad Request: {err_body[:500]}",
                        "endpoint": endpoint,
                        "payload": payload,
                        "api_error": err_json,
                    }

                if status_code == 429 or 500 <= status_code < 600:
                    if attempt < self.MAX_RETRIES:
                        backoff = 2 ** (attempt + 1)
                        time.sleep(backoff)
                        continue

                return {
                    "error": True,
                    "status_code": status_code,
                    "message": f"HTTP {status_code}: {err_body[:500]}",
                    "endpoint": endpoint,
                    "payload": payload,
                    "api_error": err_json,
                }

            except URLError as e:
                last_exception = e
                if attempt < self.MAX_RETRIES:
                    backoff = 2 ** (attempt + 1)
                    time.sleep(backoff)
                    continue
                return {
                    "error": True,
                    "status_code": None,
                    "message": f"URLError: {str(e)}",
                    "endpoint": endpoint,
                    "payload": payload,
                }

        return {
            "error": True,
            "status_code": None,
            "message": f"最终失败: {str(last_exception)}",
            "endpoint": endpoint,
            "payload": payload,
        }

    # ==================== 端点封装 ====================

    def _call_endpoint(self, endpoint: str, payload: dict, log_label: str = "") -> dict:
        """
        带缓存、限速、日志的通用端点调用。
        """
        start_time = time.time()
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]
        cache_key = self._make_cache_key(endpoint, payload)

        with self.cache_lock:
            if cache_key in self.cache:
                elapsed_ms = (time.time() - start_time) * 1000
                self._write_log(timestamp, log_label or endpoint, True, elapsed_ms)
                if self.state is not None:
                    self.state["api_stats"]["cache_hits"] += 1
                return self.cache[cache_key]

        self._acquire_token()

        result = self._do_request(endpoint, payload)

        elapsed_ms = (time.time() - start_time) * 1000

        # 更新统计
        if self.state is not None:
            if result.get("error"):
                self.state["api_stats"]["errors"] += 1
            else:
                self.state["api_stats"]["total_calls"] += 1

        # 记录日志
        extra = ""
        if result.get("error"):
            extra = f"ERROR: {result.get('message', '')[:200]}"
        self._write_log(timestamp, log_label or endpoint, False, elapsed_ms, extra)

        # 缓存成功响应
        if not result.get("error"):
            with self.cache_lock:
                self.cache[cache_key] = result
                self._save_cache()

        return result

    def meta_search(self, query: str, filters: list = None, fields: list = None,
                    page: int = 1, page_size: int = 5) -> dict:
        payload = {
            "query": query,
            "page": page,
            "page_size": page_size,
        }
        if filters is not None:
            payload["filters"] = filters
        if fields is not None:
            payload["fields"] = fields
        return self._call_endpoint("meta-search", payload, log_label=f"meta-search:{query[:40]}")

    # 保留 search 别名，兼容旧代码（如 Cell 6 GA）
    def search(self, query: str, filters: list = None, fields: list = None,
               page: int = 1, page_size: int = 5) -> dict:
        return self.meta_search(query, filters, fields, page, page_size)

    def agentic_search(self, query: str, page_size: int = 5) -> dict:
        payload = {
            "query": query,
            "page": 1,
            "page_size": page_size,
        }
        return self._call_endpoint("agentic-search", payload, log_label=f"agentic:{query[:40]}")

    def content(self, doc_id: str) -> dict:
        payload = {"doc_id": doc_id}
        return self._call_endpoint("content", payload, log_label=f"content:{doc_id[:40]}")

    def paper_relations(self, unique_id: str, relation_type: str, page_size: int = 10) -> dict:
        payload = {
            "unique_id": unique_id,
            "relation_type": relation_type,
            "page_size": page_size,
        }
        return self._call_endpoint(
            "meta-paper-relations",
            payload,
            log_label=f"relations:{relation_type}:{unique_id[:40]}"
        )

    def test_connection(self, endpoint: str = "meta-search") -> dict:
        if endpoint not in self.BASE_URLS:
            return {
                "error": True,
                "message": f"未知端点: {endpoint}，可用: {list(self.BASE_URLS.keys())}"
            }
        # 用最小请求测试连通性
        if endpoint in ("meta-search", "agentic-search"):
            payload = {"query": "test", "page": 1, "page_size": 1}
        elif endpoint == "content":
            payload = {"doc_id": "test"}
        elif endpoint == "meta-paper-relations":
            payload = {"unique_id": "test", "relation_type": "related", "page_size": 1}
        else:
            payload = {}
        return self._call_endpoint(endpoint, payload, log_label=f"test:{endpoint}")

# ==================== 测试函数 ====================

def test_rate_limit_and_cache():
    client = SciverseClient(api_token=CONFIG["sciverse_token"], state=state)
    queries = [
        "graphene battery cycle stability",
        "lithium sulfur battery cathode",
        "solid state electrolyte interface",
        "sodium ion battery anode material",
        "perovskite solar cell degradation",
    ]
    print("=" * 60)
    print("开始测试：连续调用30次，验证限速与缓存")
    print("=" * 60)

    call_count = 0
    overall_start = time.time()

    for query in queries:
        for i in range(6):
            call_count += 1
            t0 = time.time()
            try:
                result = client.meta_search(
                    query=query,
                    filters=[{
                        "field": "publication_published_year",
                        "operator": "FILTER_OP_GTE",
                        "value": 2022
                    }],
                    page=1,
                    page_size=5,
                )
                elapsed = time.time() - t0
                likely_cached = elapsed < 0.05
                print(f"  [{call_count:02d}] query='{query[:30]}...' "
                      f"耗时={elapsed:.3f}s {'(缓存)' if likely_cached else '(API)'}")
            except Exception as e:
                elapsed = time.time() - t0
                print(f"  [{call_count:02d}] query='{query[:30]}...' "
                      f"耗时={elapsed:.3f}s ERROR: {e}")

    total_elapsed = time.time() - overall_start
    print("=" * 60)
    print(f"测试完成！总调用: {call_count}次, 总耗时: {total_elapsed:.2f}秒")
    print(f"详细日志请查看: {SciverseClient.LOG_FILE}")
    print(f"缓存文件请查看: {SciverseClient.CACHE_FILE}")
    print("=" * 60)
RUN_STRESS_TEST = False

if RUN_STRESS_TEST:
    test_rate_limit_and_cache()   

     
sciverse_client = SciverseClient(api_token=CONFIG["sciverse_token"], state=state)

with open("agent_state.json", "w", encoding="utf-8") as f:
    json.dump(state, f, ensure_ascii=False, indent=2)


In [ ]:
# -*- coding: utf-8 -*-
# Cell 4: LLM 配置
# [迁移改动] 所有 LLM 配置从 CONFIG 读取，不再硬编码

# 原版: api_key="sk-xxx", base_url='https://api.deepseek.com/v1', model="deepseek-v4-flash"

# 现在统一从 CONFIG 字典读取
# -*- coding: utf-8 -*-
# Cell 4: LLM 配置

import os
from openai import OpenAI


# ============================================================
# 1. 检查前置配置
# ============================================================

if "CONFIG" not in globals():
    raise RuntimeError("请先运行 Cell 0，创建 CONFIG 配置。")

required_config_keys = [
    "deepseek_key",
    "llm_base_url",
    "llm_model",
]

missing_config_keys = [
    key for key in required_config_keys
    if not CONFIG.get(key)
]

if missing_config_keys:
    raise RuntimeError(
        f"CONFIG 缺少配置项：{missing_config_keys}"
    )


# ============================================================
# 2. 加载 System Prompt
# ============================================================

try:
    with open(
        "materials_agent_prompt.txt",
        "r",
        encoding="utf-8",
    ) as prompt_file:
        SYSTEM_PROMPT = prompt_file.read()

    print(
        f"✅ 已加载 System Prompt："
        f"{len(SYSTEM_PROMPT)} 个字符"
    )

except FileNotFoundError:
    SYSTEM_PROMPT = (
        "你是一个材料科学研究助手。"
        "你的任务是基于学术文献进行分析，"
        "识别 Research Gap，构建证据链，"
        "并输出结构化 JSON 结果。"
        "请始终使用搜索工具获取文献证据，"
        "不要编造不存在的文献、DOI 或实验数据。"
    )

    print(
        "⚠️ 没有找到 materials_agent_prompt.txt，"
        "暂时使用内置 System Prompt。"
    )


# ============================================================
# 3. 创建 DeepSeek 客户端
# ============================================================

llm_client = OpenAI(
    api_key=CONFIG["deepseek_key"].strip(),
    base_url=CONFIG["llm_base_url"],
)

# 保留旧代码可能使用的 model 变量
model = CONFIG["llm_model"]

print("✅ DeepSeek 客户端创建完成")


# ============================================================
# 4. 检查配置
# ============================================================

_diag_key = CONFIG["deepseek_key"].strip()

print("\n" + "=" * 60)
print("🔍 DeepSeek API 接入诊断")
print("=" * 60)

print(
    f"[1] API Key："
    f"{_diag_key[:4]}****{_diag_key[-3:]}"
)

print(
    f"[2] Base URL："
    f"{CONFIG['llm_base_url']}"
)

print(
    f"[3] Model："
    f"{CONFIG['llm_model']}"
)


# ============================================================
# 5. 发送一次最小连接测试
# ============================================================

if len(_diag_key) < 10:
    raise RuntimeError(
        "DeepSeek API Key 长度异常，"
        "请重新运行 Cell 0 并输入有效密钥。"
    )

try:
    print("[4] 正在发送最小连接测试……")

    _resp = llm_client.chat.completions.create(
        model=CONFIG["llm_model"],
        messages=[
            {
                "role": "user",
                "content": "请只回复四个字：连接成功",
            }
        ],
        max_tokens=50,

        # 首次调试关闭思考模式
        extra_body={
            "thinking": {
                "type": "disabled"
            }
        },
    )

    _content = (
        _resp.choices[0].message.content or ""
    ).strip()

    if _content:
        print(
            f"✅ LLM 连接成功，模型回复：{_content!r}"
        )
    else:
        print(
            "⚠️ API 请求成功，但模型返回了空文本。"
        )
        print(
            "请保留本输出，暂时不要运行后续 Agent。"
        )

except Exception as error:
    print(
        f"❌ DeepSeek 连接失败："
        f"{type(error).__name__}: {error}"
    )

    raise

print("=" * 60)
print("✅ Cell 4 执行完成")

In [ ]:
# -*- coding: utf-8 -*-
# Cell 5: 工具注册与搜索函数 (v5.1.1)
# - 适配 Sciverse 4 端点：meta-search / agentic-search / content / meta-paper-relations
# - 修复：移除所有 is_content_accessible；搜索相关工具基于 unique_id；新增 4 个评估工具

import json
import re

# ============================================================
# 工具注册系统 - 装饰器模式
# ============================================================

TOOL_REGISTRY = {}

def register_tool(description: str, parameters: dict):
    """
    工具注册装饰器 - 唯一注册入口

    用法：在 def 函数上方加 @register_tool(description="...", parameters={...})
    """
    def decorator(func):
        func._tool_schema = {
            "type": "function",
            "function": {
                "name": func.__name__,
                "description": description,
                "parameters": parameters
            }
        }
        TOOL_REGISTRY[func.__name__] = {
            "func": func,
            "schema": func._tool_schema,
            "description": description
        }
        return func
    return decorator

def get_tools_list() -> list:
    """生成给 LLM 的 tools 参数列表"""
    return [tool["schema"] for tool in TOOL_REGISTRY.values()]

# ============================================================
# 通用 JSON 解析辅助
# ============================================================

def _safe_parse_llm_json(text: str) -> dict:
    """从 LLM 输出中安全解析 JSON"""
    cleaned = text.strip()
    if cleaned.startswith("```"):
        lines = cleaned.split("\n")
        start = 1
        end = len(lines)
        if lines[-1].strip() == "```":
            end = -1
        cleaned = "\n".join(lines[start:end]).strip()
    try:
        return json.loads(cleaned)
    except Exception:
        for pattern in [r'\[[\s\S]*\]', r'\{[\s\S]*\}']:
            match = re.search(pattern, cleaned)
            if match:
                try:
                    return json.loads(match.group())
                except Exception:
                    continue
    return {"parse_error": True, "raw": text[:500]}

# ============================================================
# 通用响应提取辅助
# ============================================================

def _extract_papers(result: dict) -> list:
    """
    统一从 Sciverse 各端点响应中提取论文/段落列表。
    兼容 results / data / hits / paragraphs / chunks 等常见 key。
    """
    if not isinstance(result, dict):
        return []
    for key in ("results", "data", "hits", "papers", "paragraphs", "chunks", "items"):
        items = result.get(key)
        if isinstance(items, list):
            return items
    # 如果响应本身就是列表
    if isinstance(result, list):
        return result
    return []

def _normalize_paper(paper: dict) -> dict:
    """统一论文字段名，方便上层工具使用。"""
    if not isinstance(paper, dict):
        return paper
    return {
        "unique_id": paper.get("unique_id") or paper.get("id") or paper.get("_id"),
        "doc_id": paper.get("doc_id"),
        "title": paper.get("title", ""),
        "abstract": paper.get("abstract", ""),
        "doi": paper.get("doi", ""),
        "publication_published_year": paper.get("publication_published_year") or paper.get("year"),
        "publication_venue_name_unified": paper.get("publication_venue_name_unified", ""),
        "author": paper.get("author") or paper.get("authors", []),
        "citation_count": paper.get("citation_count") or paper.get("citations_count"),
        "source_payload": paper,
    }

def _meta_search_lookup(identifier: str, page_size: int = 5) -> list:
    """用 DOI/title 等 identifier 反查 unique_id，返回论文列表。"""
    try:
        res = sciverse_client.meta_search(query=identifier, page=1, page_size=page_size)
        if res.get("error"):
            return []
        papers = _extract_papers(res)
        return [_normalize_paper(p) for p in papers]
    except Exception:
        return []

# ============================================================
# 工具 1: 文献强检索
# ============================================================

@register_tool(
    description="【强检索】在 Sciverse 学术数据库中搜索研究文献。"
    "适用于需要查找特定领域最新论文、已知关键词的精确搜索、验证假设。"
    "query 应使用专业术语和精确关键词，page_size 默认 5 适合聚焦搜索。"
    "如果需要扩大搜索范围、寻找跨领域关联，请配合 search_weak 使用。"
    "返回文献列表，包含 unique_id、title、abstract、doi、year、venue 等字段。默认获取摘要，不获取全文。",
    parameters={
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "研究主题或关键词，如 'perovskite solar cell stability'"
            },
            "year_min": {
                "type": "integer",
                "description": "筛选发表年份下限（含），如 2022。不填则不限"
            },
            "page_size": {
                "type": "integer",
                "description": "返回文献条数，默认 5。聚焦搜索建议 5，发散搜索建议 10"
            }
        },
        "required": ["query"]
    }
)
def search_literature(query: str, year_min: int = None, page_size: int = 5):
    """
    在 Sciverse 学术数据库中搜索研究文献（强检索工具）
    - meta-search 默认返回字段已包含 abstract，无需传自定义 fields
    """
    try:
        filters = None
        if year_min:
            filters = [{
                "field": "publication_published_year",
                "operator": "FILTER_OP_GTE",
                "value": year_min
            }]

        result = sciverse_client.meta_search(
            query=query,
            filters=filters,
            page=1,
            page_size=page_size
        )

        if result.get("error"):
            return {
                "status": "error",
                "query": query,
                "error": result.get("message", "未知错误"),
                "api_error": result.get("api_error")
            }

        papers = [_normalize_paper(p) for p in _extract_papers(result)]
        return {
            "status": "success",
            "query": query,
            "year_min": year_min,
            "total_returned": len(papers),
            "papers": papers,
            "raw": result
        }

    except Exception as e:
        return {"status": "error", "query": query, "error": str(e)}

# ============================================================
# 工具 2: 弱检索
# ============================================================

@register_tool(
    description="【弱检索】扩大搜索范围的学术文献检索工具。"
    "当需要扩大搜索范围、寻找跨领域关联、发现被遗漏的相关工作时使用。"
    "与 search_literature 的区别：使用近义词、上位概念、跨领域关键词，返回更多结果。"
    "适用场景：(1) 强检索结果太少 (<3条)；(2) 需要发现跨领域灵感；"
    "(3) 需要找到被不同术语描述的相同概念。",
    parameters={
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "宽泛关键词，建议使用上位概念或近义词。"
            },
            "year_min": {
                "type": "integer",
                "description": "筛选发表年份下限（含），如 2020。不填则不限"
            },
            "page_size": {
                "type": "integer",
                "description": "返回文献条数，默认 10（弱检索需要更大范围）。最大 20"
            }
        },
        "required": ["query"]
    }
)
def search_weak(query: str, year_min: int = None, page_size: int = 10):
    """弱检索工具 - 使用宽泛/跨领域/近义词关键词扩大搜索范围"""
    page_size = min(page_size, 20)
    try:
        filters = None
        if year_min:
            filters = [{
                "field": "publication_published_year",
                "operator": "FILTER_OP_GTE",
                "value": year_min
            }]

        result = sciverse_client.meta_search(
            query=query,
            filters=filters,
            page=1,
            page_size=page_size
        )

        if result.get("error"):
            return {
                "status": "error",
                "query": query,
                "error": result.get("message", "未知错误"),
                "api_error": result.get("api_error")
            }

        papers = [_normalize_paper(p) for p in _extract_papers(result)]
        return {
            "status": "success",
            "query": query,
            "search_type": "weak",
            "year_min": year_min,
            "total_returned": len(papers),
            "papers": papers,
            "raw": result
        }

    except Exception as e:
        return {"status": "error", "query": query, "search_type": "weak", "error": str(e)}

# ============================================================
# 工具 3: 关联搜索（引用/被引/相关）
# ============================================================

@register_tool(
    description="【关联搜索】基于 unique_id 追踪论文的引用/被引/相关工作。"
    "当你已知某篇关键论文的 unique_id 时使用；如果只有 DOI 或 title，本工具会先反查 unique_id。"
    "这是真正的引用关系检索，不是关键词模拟。",
    parameters={
        "type": "object",
        "properties": {
            "unique_id": {
                "type": "string",
                "description": "来自 meta-search 的论文 unique_id。优先使用此参数。"
            },
            "doi": {
                "type": "string",
                "description": "论文 DOI（当 unique_id 未知时用于反查）。"
            },
            "title": {
                "type": "string",
                "description": "论文标题（当 unique_id 和 DOI 都未知时用于反查）。"
            },
            "relation_type": {
                "type": "string",
                "enum": ["citations", "references", "related"],
                "description": "关系类型：'citations'=谁引用了这篇论文，'references'=这篇论文引用了谁，'related'=相关工作。"
            },
            "page_size": {
                "type": "integer",
                "description": "返回结果数量，默认 10"
            }
        },
        "required": ["relation_type"]
    }
)
def search_related(unique_id: str = None, doi: str = None, title: str = None,
                   relation_type: str = "related", page_size: int = 10):
    """
    关联搜索工具 - 基于 meta-paper-relations 追踪引用/被引/相关工作
    - 如果未提供 unique_id，先用 DOI 或 title 反查
    """
    # 1. 确定 unique_id
    target_id = unique_id
    if not target_id:
        lookup = doi or title
        if not lookup:
            return {"status": "error", "message": "必须提供 unique_id、doi 或 title 至少一个参数"}
        papers = _meta_search_lookup(lookup, page_size=3)
        if not papers:
            return {"status": "error", "message": f"无法通过 '{lookup}' 找到 unique_id"}
        target_id = papers[0].get("unique_id")
        if not target_id:
            return {"status": "error", "message": "反查成功但结果中缺少 unique_id"}

    # 2. 标准化 relation_type
    relation_type = relation_type.lower().strip()
    allowed = {"citations", "references", "related"}
    if relation_type not in allowed:
        relation_type = "related"

    try:
        result = sciverse_client.paper_relations(
            unique_id=target_id,
            relation_type=relation_type,
            page_size=page_size
        )

        if result.get("error"):
            return {
                "status": "error",
                "unique_id": target_id,
                "relation_type": relation_type,
                "error": result.get("message", "未知错误"),
                "api_error": result.get("api_error")
            }

        papers = [_normalize_paper(p) for p in _extract_papers(result)]
        return {
            "status": "success",
            "unique_id": target_id,
            "relation_type": relation_type,
            "total_returned": len(papers),
            "papers": papers,
            "raw": result
        }

    except Exception as e:
        return {
            "status": "error",
            "unique_id": target_id,
            "relation_type": relation_type,
            "error": str(e)
        }

# ============================================================
# 工具 4: 按需获取单篇文献内容
# ============================================================

@register_tool(
    description="【按需内容获取】针对已筛选出的高价值单篇文献，获取其摘要、相关段落或全文。"
    "在文献调研的前几轮（发散/收敛阶段）应该先通过 search_literature/search_weak/search_related 获取批量题录+摘要；"
    "只有当某篇文献对验证假设、填补证据链至关重要时，才调用本工具获取更完整内容。"
    "参数 level 控制获取深度：abstract 仅摘要；paragraphs 返回相关段落（agentic-search）；"
    "fulltext 尝试返回全文（需要该论文有 doc_id）。",
    parameters={
        "type": "object",
        "properties": {
            "unique_id": {
                "type": "string",
                "description": "来自 meta-search 的论文 unique_id。优先使用。"
            },
            "doi": {
                "type": "string",
                "description": "文献 DOI，用于反查 unique_id。"
            },
            "title": {
                "type": "string",
                "description": "文献标题，当 unique_id/DOI 不可用时使用。"
            },
            "level": {
                "type": "string",
                "enum": ["abstract", "paragraphs", "fulltext"],
                "description": "获取深度。'abstract'=摘要（成本低）；'paragraphs'=相关段落；'fulltext'=全文（需要 doc_id，否则降级为 paragraphs）。"
            }
        },
        "required": ["level"]
    }
)
def fetch_paper_content(unique_id: str = None, doi: str = None, title: str = None,
                        level: str = "abstract"):
    """
    按需获取单篇文献的摘要、段落或全文。
    - abstract: 直接返回 meta_search 的 abstract
    - paragraphs: 调用 agentic_search
    - fulltext: 先拿 doc_id，再调 content；无 doc_id 则降级为 paragraphs
    """
    if not unique_id and not doi and not title:
        return {"status": "error", "message": "必须提供 unique_id、doi 或 title 至少一个参数"}

    # 1. 获取论文元数据
    paper = None
    if unique_id:
        papers = _meta_search_lookup(unique_id, page_size=3)
        if papers:
            paper = papers[0]
    if not paper and (doi or title):
        papers = _meta_search_lookup(doi or title, page_size=3)
        if papers:
            paper = papers[0]

    if not paper:
        return {
            "status": "no_results",
            "unique_id": unique_id,
            "doi": doi,
            "title": title,
            "level": level,
            "message": "无法通过给定标识定位到论文。"
        }

    # 2. abstract 级别
    if level == "abstract":
        return {
            "status": "success",
            "level": "abstract",
            "unique_id": paper.get("unique_id"),
            "doc_id": paper.get("doc_id"),
            "title": paper.get("title"),
            "doi": paper.get("doi"),
            "year": paper.get("publication_published_year"),
            "venue": paper.get("publication_venue_name_unified"),
            "abstract": paper.get("abstract", ""),
            "note": "仅返回摘要。"
        }

    # 3. paragraphs 级别
    if level == "paragraphs":
        try:
            res = sciverse_client.agentic_search(
                query=paper.get("title", doi or title),
                page_size=5
            )
            if res.get("error"):
                return {
                    "status": "error",
                    "level": "paragraphs",
                    "unique_id": paper.get("unique_id"),
                    "error": res.get("message", "未知错误"),
                    "api_error": res.get("api_error")
                }
            paragraphs = _extract_papers(res)
            return {
                "status": "success",
                "level": "paragraphs",
                "unique_id": paper.get("unique_id"),
                "doc_id": paper.get("doc_id"),
                "title": paper.get("title"),
                "doi": paper.get("doi"),
                "paragraphs": paragraphs,
                "raw": res
            }
        except Exception as e:
            return {"status": "error", "level": "paragraphs", "error": str(e)}

    # 4. fulltext 级别
    if level == "fulltext":
        doc_id = paper.get("doc_id")
        if doc_id:
            try:
                res = sciverse_client.content(doc_id=doc_id)
                if res.get("error"):
                    # 出错则降级为 paragraphs
                    fallback = fetch_paper_content(
                        unique_id=paper.get("unique_id"),
                        level="paragraphs"
                    )
                    fallback["note"] = f"content 端点失败({res.get('message', '')})，已降级为 paragraphs。"
                    return fallback
                return {
                    "status": "success",
                    "level": "fulltext",
                    "unique_id": paper.get("unique_id"),
                    "doc_id": doc_id,
                    "title": paper.get("title"),
                    "doi": paper.get("doi"),
                    "content": res,
                    "note": "返回全文内容。"
                }
            except Exception as e:
                fallback = fetch_paper_content(
                    unique_id=paper.get("unique_id"),
                    level="paragraphs"
                )
                fallback["note"] = f"content 端点异常({str(e)})，已降级为 paragraphs。"
                return fallback
        else:
            # 无 doc_id，降级为 paragraphs
            fallback = fetch_paper_content(
                unique_id=paper.get("unique_id"),
                level="paragraphs"
            )
            fallback["note"] = "该论文没有 doc_id，已降级为 paragraphs。"
            return fallback

    return {"status": "error", "message": f"未知 level: {level}"}

# ============================================================
# 辅助函数: 关键词提取（内部使用）
# ============================================================

def extract_keywords(search_results_text: str) -> list:
    """
    从搜索结果文本中提取关键词/实体 - 内部辅助函数
    """
    keywords = []

    title_pattern = r'"title"\s*:\s*"([^"]+)"'
    titles = re.findall(title_pattern, search_results_text)
    for t in titles[:5]:
        if len(t) > 10:
            keywords.append({"type": "title", "value": t})

    year_pattern = r'"publication_published_year"\s*:\s*(\d{4})'
    years = re.findall(year_pattern, search_results_text)
    unique_years = list(set(years))[:3]
    for y in unique_years:
        keywords.append({"type": "year", "value": y})

    doi_pattern = r'"doi"\s*:\s*"([^"]+)"'
    dois = re.findall(doi_pattern, search_results_text)
    for doi in dois[:3]:
        keywords.append({"type": "doi", "value": doi})

    concept_pattern = r'"([^"]{4,50})"'
    concepts = re.findall(concept_pattern, search_results_text)
    skip_words = {"title", "doi", "status", "query", "results", "error",
                  "success", "publication_published_year", "type", "value",
                  "publication_venue_name_unified", "search_type", "entity",
                  "relation_type", "year_min", "page_size", "fields"}
    for c in concepts[:10]:
        if c.lower() not in skip_words and not c.startswith("sci_") and not c.startswith("sk-"):
            keywords.append({"type": "concept", "value": c})

    return keywords[:20]

# ============================================================
# 工具 5: 可证伪性检查
# ============================================================

@register_tool(
    description="对当前最有可能的假设/结论进行可证伪性检查。"
    "通过模拟审稿人视角，识别潜在漏洞、分析证据反驳力、建议验证实验，"
    "最终判定该假设是否可证伪。"
    "仅在搜索阶段充分后、准备输出最终结论前调用此工具。",
    parameters={
        "type": "object",
        "properties": {
            "hypothesis": {
                "type": "string",
                "description": "需要检验的假设或结论，应基于前期搜索证据提出"
            },
            "evidence_summary": {
                "type": "string",
                "description": "支持该假设的证据摘要（不超过300字），来自前期搜索"
            }
        },
        "required": ["hypothesis", "evidence_summary"]
    }
)
def falsification_check(hypothesis: str, evidence_summary: str):
    """可证伪性检查工具"""
    try:
        prompt = f"""
你是一位严格的学术审稿人。请对以下假设进行可证伪性分析。

【待检验的假设】
{hypothesis}

【支持证据摘要】
{evidence_summary}

请按以下格式输出分析结果：

潜在漏洞：
1. [漏洞1的具体描述]
2. [漏洞2的具体描述]
3. [漏洞3的具体描述]

证据反驳分析：
- 漏洞1：[该漏洞在多大程度上可以被现有证据反驳？]
- 漏洞2：[该漏洞在多大程度上可以被现有证据反驳？]
- 漏洞3：[该漏洞在多大程度上可以被现有证据反驳？]

所需验证实验：
[列出需要通过什么实验来进一步验证这些漏洞]

最终判定：
[从以下三个选项中选择一个：可证伪 / 部分可证伪 / 不可证伪]
"""
        response = llm_client.chat.completions.create(
            model=CONFIG["llm_model"],
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            extra_body={"thinking": {"type": "disabled"}},
        )
        raw_output = response.choices[0].message.content.strip()
        return _parse_falsification_output(raw_output)
    except Exception as e:
        return {
            "status": "parse_failed",
            "raw": f"Error during LLM call or parsing: {str(e)}"
        }

def _parse_falsification_output(text: str) -> dict:
    try:
        text = re.sub(r'(?m)^\s*#{1,6}\s*', '', text)
        text = text.replace('**', '')
        vulnerabilities_match = re.search(r'潜在漏洞：\s*\n((?:\d+\.\s*.+\n?)+)', text)
        rebuttal_match = re.search(r'证据反驳分析：\s*\n((?:-\s*漏洞\d+：.+\n?)+)', text)
        experiment_match = re.search(r'所需验证实验：\s*\n(.+?)(?=\n最终判定：)', text, re.DOTALL)
        verdict_match = re.search(r'最终判定：\s*\n\s*(可证伪|部分可证伪|不可证伪)', text)

        vulnerabilities = []
        if vulnerabilities_match:
            vuln_text = vulnerabilities_match.group(1)
            vulnerabilities = [line.split('.', 1)[1].strip()
                             for line in vuln_text.strip().split('\n')
                             if line.strip()]

        rebuttals = {}
        if rebuttal_match:
            reb_text = rebuttal_match.group(1)
            for line in reb_text.strip().split('\n'):
                if '：' in line:
                    key, value = line.split('：', 1)
                    vuln_num = re.search(r'\d+', key)
                    if vuln_num:
                        rebuttals[f"漏洞{vuln_num.group()}"] = value.strip()

        return {
            "status": "success",
            "vulnerabilities": vulnerabilities[:3],
            "rebuttals": rebuttals,
            "required_experiments": experiment_match.group(1).strip() if experiment_match else "",
            "verdict": verdict_match.group(1) if verdict_match else "未知"
        }
    except Exception:
        return {"status": "parse_failed", "raw": text}

# ============================================================
# 工具 6: DOI 断言验证（v5.1.1 增强版）
# ============================================================

@register_tool(
    description="【DOI验证】验证一个 DOI 是否真实存在，并判断给定断言是否被该论文支持。"
    "当某条证据声称来自某 DOI 时，调用本工具确认：1) DOI 是否真实存在；2) 该断言是否被论文内容支持。"
    "返回状态包括 verified / partial / unsupported / not_found。",
    parameters={
        "type": "object",
        "properties": {
            "doi": {
                "type": "string",
                "description": "待验证的 DOI，例如 '10.1038/nmat1718'"
            },
            "claim": {
                "type": "string",
                "description": "要验证的断言。"
            }
        },
        "required": ["doi", "claim"]
    }
)
def verify_doi_reference(doi: str, claim: str):
    """
    1. 用 DOI 调 meta_search 确认存在
    2. 用 fetch_paper_content(level='paragraphs') 获取段落
    3. 让 LLM 判断 claim 是否被支持
    """
    if not doi or not doi.startswith("10."):
        return {"status": "invalid", "doi": doi, "message": "DOI 格式异常，应以 10. 开头"}

    # 1. 确认 DOI 存在并获取元数据
    try:
        res = sciverse_client.meta_search(query=doi, page=1, page_size=3)
        if res.get("error"):
            return {
                "status": "error",
                "doi": doi,
                "message": f"Sciverse 请求失败: {res.get('message', '')}"
            }
        papers = _extract_papers(res)
        if not papers:
            return {"status": "not_found", "doi": doi, "message": "Sciverse 未返回该 DOI 对应的论文"}
        paper = _normalize_paper(papers[0])
        matched_doi = paper.get("doi", "")
        if matched_doi != doi:
            return {
                "status": "not_found",
                "doi": doi,
                "matched_doi": matched_doi,
                "message": "未精确匹配到该 DOI"
            }
    except Exception as e:
        return {"status": "error", "doi": doi, "message": f"DOI 查找异常: {str(e)}"}

    # 2. 获取段落内容
    paragraphs_result = fetch_paper_content(
        unique_id=paper.get("unique_id"),
        level="paragraphs"
    )
    paragraphs = ""
    if paragraphs_result.get("status") == "success":
        paras = paragraphs_result.get("paragraphs", [])
        if paras:
            paragraphs = "\n\n".join([
                p.get("body", p.get("text", str(p))) for p in paras[:5]
            ])

    # 3. LLM 判断
    try:
        prompt = f"""
你是一位严格的文献核对专家。请根据以下论文信息，判断给定断言是否被该论文支持。

论文标题：{paper.get('title', '')}
论文摘要：{paper.get('abstract', '')}
论文段落：{paragraphs}

待验证断言：{claim}

请按以下格式输出：
- verdict: [supported / partially_supported / unsupported / not_found]
- 理由：...
- 引用的原文片段：...
- 注意：如果论文中完全没有提到断言中的关键概念，必须返回 not_found
"""
        resp = llm_client.chat.completions.create(
            model=CONFIG["llm_model"],
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            max_tokens=1024,
            extra_body={"thinking": {"type": "disabled"}},
        )
        llm_text = resp.choices[0].message.content.strip()

        # 解析 verdict
        verdict = "not_found"
        if "partially_supported" in llm_text.lower() or "partial" in llm_text.lower():
            verdict = "partial"
        elif "supported" in llm_text.lower():
            verdict = "verified"
        elif "unsupported" in llm_text.lower():
            verdict = "unsupported"

        # 状态映射
        status_map = {
            "verified": "verified",
            "partial": "partial",
            "unsupported": "unsupported",
            "not_found": "not_found"
        }

        # 尝试提取原文片段
        evidence_quote = ""
        quote_match = re.search(r'引用的原文片段[：:]\s*(.+?)(?=\n|$)', llm_text, re.DOTALL)
        if quote_match:
            evidence_quote = quote_match.group(1).strip()

        return {
            "status": status_map.get(verdict, "not_found"),
            "doi": doi,
            "paper_title": paper.get("title", ""),
            "verdict": "断言被摘要/正文支持" if verdict == "verified" else
                       "断言被部分支持" if verdict == "partial" else
                       "断言与论文内容不符" if verdict == "unsupported" else
                       "无法确认",
            "evidence_quote": evidence_quote,
            "llm_analysis": llm_text[:800]
        }
    except Exception as e:
        return {
            "status": "error",
            "doi": doi,
            "paper_title": paper.get("title", ""),
            "message": f"LLM 验证异常: {str(e)}"
        }

# ============================================================
# 工具 7: 断言提取
# ============================================================

@register_tool(
    description="【断言提取】从论文摘要或段落文本中，提取与某个假设相关的结构化断言。"
    "用于把非结构化的论文文本变成可验证的 claim 列表，每个 claim 标注置信度和相关实体。",
    parameters={
        "type": "object",
        "properties": {
            "paper_text": {
                "type": "string",
                "description": "论文摘要或段落文本"
            },
            "hypothesis": {
                "type": "string",
                "description": "可选，用于筛选相关 claim 的假设或研究问题"
            }
        },
        "required": ["paper_text"]
    }
)
def extract_claims(paper_text: str, hypothesis: str = ""):
    """用 LLM 从文本中提取与假设相关的结构化断言。"""
    try:
        prompt = f"""
请从以下论文内容中提取具体的事实断言。每个断言必须是论文明确表达的内容，不能是你自己的推断。

论文内容：
{paper_text[:4000]}

{"相关假设：" + hypothesis if hypothesis else ""}

输出格式：JSON 数组
[
  {{
    "claim_text": "论文中的断言",
    "entity": "涉及的材料/对象",
    "property": "性能/性质",
    "value": "数值（如有）",
    "unit": "单位（如有）",
    "confidence": "high/medium/low"
  }}
]

注意：
- 只提取论文中明确陈述的事实
- 数值必须带单位
- 如果某个断言只是背景介绍，不是研究发现，confidence 设为 low
"""
        resp = llm_client.chat.completions.create(
            model=CONFIG["llm_model"],
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            max_tokens=1024,
            extra_body={"thinking": {"type": "disabled"}},
        )
        content = resp.choices[0].message.content
        parsed = _safe_parse_llm_json(content)
        if isinstance(parsed, list):
            return {"status": "success", "hypothesis": hypothesis, "claims": parsed}
        elif "parse_error" in parsed:
            return {"status": "parse_failed", "hypothesis": hypothesis, "raw": parsed.get("raw", "")}
        else:
            return {"status": "success", "hypothesis": hypothesis, "claims": [parsed]}
    except Exception as e:
        return {"status": "error", "hypothesis": hypothesis, "error": str(e)}

# ============================================================
# 工具 8: 论文相关性过滤
# ============================================================

@register_tool(
    description="【相关性过滤】让 LLM 判断单篇论文与研究问题的相关程度。"
    "返回 topic/method/conclusion 三个维度的评分和综合判断。",
    parameters={
        "type": "object",
        "properties": {
            "paper": {
                "type": "object",
                "description": "来自 meta_search 的单篇论文（包含 title、abstract 等字段）。"
            },
            "research_question": {
                "type": "string",
                "description": "研究问题或假设"
            },
            "criteria": {
                "type": "array",
                "items": {"type": "string"},
                "description": '评估维度，如 ["topic", "method", "conclusion"]'
            }
        },
        "required": ["paper", "research_question"]
    }
)
def filter_paper_relevance(paper: dict, research_question: str, criteria: list = None):
    """让 LLM 判断论文与研究问题的相关性。"""
    if not isinstance(paper, dict):
        return {"status": "error", "message": "paper 必须是 dict"}
    criteria = criteria or ["topic", "method", "conclusion"]
    title = paper.get("title", "")
    abstract = paper.get("abstract", "")
    try:
        prompt = f"""
请判断以下论文与给定研究问题的相关性。

研究问题：{research_question}
论文标题：{title}
论文摘要：{abstract}

请从以下维度打分（0-1）：
1. topic_relevance：论文主题是否与研究问题相关
2. method_relevance：论文方法是否可用于回答研究问题
3. conclusion_relevance：论文结论是否直接支持/反驳研究问题

输出格式（严格 JSON）：
{{
  "is_relevant": true/false,
  "overall_score": 0.0-1.0,
  "topic_score": 0.0-1.0,
  "method_score": 0.0-1.0,
  "conclusion_score": 0.0-1.0,
  "reason": "简要说明"
}}
"""
        resp = llm_client.chat.completions.create(
            model=CONFIG["llm_model"],
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            max_tokens=512,
            extra_body={"thinking": {"type": "disabled"}},
        )
        content = resp.choices[0].message.content
        parsed = _safe_parse_llm_json(content)
        if "parse_error" in parsed:
            return {"status": "parse_failed", "raw": parsed.get("raw", "")}
        return {
            "status": "success",
            "criteria": criteria,
            "is_relevant": parsed.get("is_relevant", False),
            "score": parsed.get("overall_score", 0.0),
            "dimensions": {
                "topic": parsed.get("topic_score", 0.0),
                "method": parsed.get("method_score", 0.0),
                "conclusion": parsed.get("conclusion_score", 0.0)
            },
            "reason": parsed.get("reason", "")
        }
    except Exception as e:
        return {"status": "error", "error": str(e)}

# ============================================================
# 工具 9: 机制一致性检查（v5.1.1 替换原 mechanism_validator）
# ============================================================

@register_tool(
    description="【机制验证】验证给定假设在指定学科领域内的机制合理性。"
    "不依赖外部数据库，基于 LLM 的领域知识判断假设的物理/化学/学科机制是否自洽。"
    "适用于快速判断一个构效关系假设是否值得进一步验证。",
    parameters={
        "type": "object",
        "properties": {
            "hypothesis": {
                "type": "string",
                "description": "待验证的假设或构效关系"
            },
            "domain": {
                "type": "string",
                "description": "学科领域，例如 materials_science, chemistry, biology, physics"
            },
            "evidence_summary": {
                "type": "string",
                "description": "支持或反驳该假设的证据摘要（不超过500字）"
            }
        },
        "required": ["hypothesis", "domain"]
    }
)
def check_mechanism_consistency(hypothesis: str, domain: str, evidence_summary: str = ""):
    """基于 LLM 和证据摘要，验证假设的机制合理性。"""
    try:
        prompt = f"""
你是 {domain} 领域的专家。请判断以下假设在科学机制上是否合理。

假设：
{hypothesis}

{f"证据摘要：{evidence_summary}" if evidence_summary else ""}

请基于该领域已知的科学原理，回答：
1. 该假设是否与已知原理一致？
2. 有哪些已知的原理支持或反驳它？
3. 可能存在什么机制上的问题？

输出格式（严格 JSON）：
{{
  "is_consistent": true/false,
  "score": 0.0-1.0,
  "supporting_principles": ["..."],
  "conflicting_principles": ["..."],
  "potential_issues": ["..."],
  "reason": "..."
}}

评分标准：
- 0.8-1.0：机制清晰，与已知原理一致，因果链完整
- 0.5-0.79：机制基本合理，但存在未解决细节
- 0.2-0.49：机制牵强，与已知原理部分冲突
- 0.0-0.19：机制明显错误或与已知原理矛盾
"""
        resp = llm_client.chat.completions.create(
            model=CONFIG["llm_model"],
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            max_tokens=1024,
            extra_body={"thinking": {"type": "disabled"}},
        )
        content = resp.choices[0].message.content
        parsed = _safe_parse_llm_json(content)
        if "parse_error" in parsed:
            return {"status": "parse_failed", "raw": parsed.get("raw", "")}
        return {
            "status": "success",
            "hypothesis": hypothesis,
            "domain": domain,
            "is_consistent": parsed.get("is_consistent", False),
            "score": parsed.get("score", 0.0),
            "supporting_principles": parsed.get("supporting_principles", []),
            "conflicting_principles": parsed.get("conflicting_principles", []),
            "potential_issues": parsed.get("potential_issues", []),
            "reason": parsed.get("reason", "")
        }
    except Exception as e:
        return {"status": "error", "hypothesis": hypothesis, "error": str(e)}

# ============================================================
# 收尾
# ============================================================

TOOLS = get_tools_list()

print(f"✅ 已注册 {len(TOOL_REGISTRY)} 个工具: {list(TOOL_REGISTRY.keys())}")
print(f"  TOOLS schema 数量: {len(TOOLS)}")

with open("agent_state.json", "w", encoding="utf-8") as f:
    json.dump(state, f, ensure_ascii=False, indent=2)


In [ ]:
# -*- coding: utf-8 -*-
# Cell 6: 遗传算法搜索 (v5.1 证据驱动评估 + 可配置搜索空间)
"""
材料构效关系遗传算法搜索框架（Sciverse集成版）

- 通过 sciverse_client 自动检索文献生成初始种群
- 复用项目已有的缓存、日志、API认证体系
- LLM评估直接使用前置Cell已初始化的 llm_client (OpenAI SDK)
- 纯标准库实现
"""

import json
import random
random.seed(int(os.environ.get("GA_SEED", "42")))
import copy
import time
import os
from typing import List, Dict, Any, Optional

# ==================== GA 专属配置 ====================

MAX_GENERATIONS = 5
POPULATION_SIZE = 10
CROSSOVER_RATE = 0.8
MUTATION_RATE = 0.3
RESULTS_FILE = os.path.join(SCIVERSE_DATA_DIR, "ga_results.json") if SCIVERSE_DATA_DIR else "results.json"

# [迁移改动] 默认文献检索 query（如果 state 中没有 gaps，则使用此默认值）

LITERATURE_QUERY = "LiFePO4 cathode doping rate performance material structure property relationship"

# 材料搜索空间
# [v5.1改动] 可从 CONFIG["ga_search_space"] 读取，保持通用性
# 如果 CONFIG 未定义，则使用默认示例（LiFePO4 掺杂体系）

DEFAULT_SEARCH_SPACE: Dict[str, list] = {
    "base_material": ["LiFePO4", "Na3V2(PO4)3"],
    "doping_element": ["Mg", "Al", "Ti", "None"],
    "doping_ratio": [0.01, 0.03, 0.05, 0.07],
    "coating_layer": ["Carbon", "Al2O3", "None"],
    "synthesis_temp_c": [600, 650, 700, 750],
    "calcination_time_h": [4, 6, 8, 10],
}

SEARCH_SPACE: Dict[str, list] = CONFIG.get("ga_search_space", DEFAULT_SEARCH_SPACE)

# [v5.1改动] 默认文献检索 query 和 topic 前缀可从 CONFIG 覆盖
LITERATURE_QUERY = CONFIG.get("ga_default_query",
    "LiFePO4 cathode doping rate performance material structure property relationship")
GA_TOPIC_PREFIX = CONFIG.get("ga_topic_prefix", "LiFePO4 cathode doping")

# ==================== 日志工具 ====================

def ga_log(message: str):
    """写入项目统一日志文件，同时打印到控制台"""
    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] [GA] {message}"
    print(line)
    if LOG_FILE_PATH:
        try:
            with open(LOG_FILE_PATH, "a", encoding="utf-8") as f:
                f.write(line + "\n")
        except Exception as e:
            print(f"  ⚠ 日志写入失败: {e}")

# ==================== JSON 解析工具 ====================

def parse_llm_json(text: str) -> Any:
    """从LLM回复中提取JSON，兼容markdown代码块和截断修复"""
    cleaned = text.strip()

    # 去除 markdown 代码块包裹
    if cleaned.startswith("```"):
        lines = cleaned.split("\n")
        # 去掉首行 ```json 和末行 ```
        start = 1
        end = len(lines)
        if lines[-1].strip() == "```":
            end = -1
        cleaned = "\n".join(lines[start:end]).strip()

    # 第一次尝试直接解析
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    # 第二次尝试：修复常见的截断问题
    # 去掉末尾不完整的行（可能是被截断的字符串）
    lines = cleaned.split("\n")
    # 从末尾往前找，去掉不完整的行
    while lines:
        last_line = lines[-1].strip()
        # 如果最后一行明显不完整（有引号没闭合、没有逗号/括号结尾）
        if last_line and not last_line.endswith(('"', ']', '}', ',', 'true', 'false', 'null')):
            lines.pop()
        else:
            break

    # 补全可能缺失的括号
    repaired = "\n".join(lines)
    # 计算未闭合的括号数
    open_braces = repaired.count("{") - repaired.count("}")
    open_brackets = repaired.count("[") - repaired.count("]")
    # 去掉末尾可能的不完整对象
    if repaired.rstrip().endswith('"'):
        repaired = repaired.rstrip().rstrip('"')

    # 尝试补全括号
    for _ in range(open_brackets):
        repaired += "\n]"
    for _ in range(open_braces):
        repaired += "\n}"

    try:
        return json.loads(repaired)
    except json.JSONDecodeError:
        pass

    # 第三次尝试：从文本中找到最大的有效JSON片段
    import re
    # 尝试找最外层的 [ ... ] 或 { ... }
    for pattern in [r'\[[\s\S]*\]', r'\{[\s\S]*\}']:
        match = re.search(pattern, cleaned)
        if match:
            try:
                return json.loads(match.group())
            except json.JSONDecodeError:
                continue

    # 全部失败，抛出明确错误
    raise ValueError(
        f"LLM 返回的内容无法解析为 JSON。\n"
        f"前200字符: {cleaned[:200]}\n"
        f"建议：增大 max_tokens 或减少请求的候选数量"
    )

# ==================== 个体表示 ====================

class Individual:
    def __init__(self, genes: Dict[str, Any]):
        self.genes = genes
        self.fitness: float = 0.0
        self.eval_reason: str = ""

    def to_dict(self) -> dict:
        return {"genes": self.genes, "fitness": self.fitness, "eval_reason": self.eval_reason}

# ==================== 遗传算法引擎 ====================

class MaterialGA:
    def __init__(self):
        # [迁移改动] 依赖检查：增加 CONFIG 检查
        if 'llm_client' not in globals() or llm_client is None:
            raise RuntimeError("❌ llm_client 未初始化，请先运行 LLM 配置 Cell")
        if 'sciverse_client' not in globals() or sciverse_client is None:
            raise RuntimeError("❌ sciverse_client 未初始化，请先运行 Sciverse Cell")
        if 'CONFIG' not in globals():
            raise RuntimeError("❌ CONFIG 未定义，请先运行 Cell 0")

        self.population: List[Individual] = []
        self.generation: int = 0
        self.history: List[dict] = []
        # [迁移改动] 使用 CONFIG["llm_model"] 替代 model 变量
        ga_log(f"GA引擎初始化成功 | 模型: {CONFIG['llm_model']} | 缓存: {CACHE_FILE_PATH}")

    def _call_llm(self, system_prompt: str, user_prompt: str) -> str:
        """统一调用入口：使用全局 llm_client"""
        # [迁移改动] 使用 CONFIG["llm_model"] 替代原来的 model 变量
        resp = llm_client.chat.completions.create(
            model=CONFIG["llm_model"],
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=1.0,
            # max_tokens=2048,#这一块地方可以设置单次调用的tok上限。
            response_format={"type": "json_object"},
            extra_body={"thinking": {"type": "disabled"}},
        )
        return resp.choices[0].message.content

    # ---------- 1. 文献驱动初始化 ----------
    def initialize_population(self):
        # [v5.1改动] 从 state["context"]["gaps"] 读取 research gaps，每个 gap 单独短 query 搜索，避免拼接导致 500
        # 使用 CONFIG 中的 GA_TOPIC_PREFIX，使 query 构造可配置、通用
        gaps = state.get("context", {}).get("gaps", [])
        all_search_results = []
        topic_prefix = GA_TOPIC_PREFIX
        if not topic_prefix and 'target_property' in state.get("context", {}):
            topic_prefix = state["context"]["target_property"]
        if not topic_prefix:
            topic_prefix = "material structure property relationship"

        if gaps:
            ga_log(f"发现 {len(gaps)} 个 Research Gaps，用于指导文献检索")
            for idx, gap in enumerate(gaps[:3], 1):  # 最多取前3个 gap
                gap_short = gap[:80]
                query = f"{topic_prefix} {gap_short}"
                ga_log(f"  [{idx}/3] 基于 Gap 搜索: '{query[:100]}...'")
                try:
                    res = sciverse_client.search(query, page_size=5)
                    all_search_results.append({"gap": gap_short, "results": res})
                except Exception as e:
                    ga_log(f"    ⚠ 该 Gap 搜索失败: {e}")
        else:
            ga_log("未发现 Research Gaps，使用默认检索 query")
            try:
                res = sciverse_client.search(LITERATURE_QUERY, page_size=5)
                all_search_results.append({"gap": "default", "results": res})
            except Exception as e:
                ga_log(f"⚠ 默认检索失败: {e}")

        self.literature_evidence = all_search_results
        literature_text = json.dumps(all_search_results, ensure_ascii=False, indent=2)
        ga_log(f"文献检索完成，共 {len(all_search_results)} 组结果")

        if not literature_text.strip() or literature_text == "[]":
            literature_text = "（文献检索不可用，请基于领域通识生成方案）"

        keys = list(SEARCH_SPACE.keys())

        # [迁移改动] 如果有 gaps，在 system prompt 中告知 LLM
        gaps_instruction = ""
        if gaps:
            gaps_instruction = (
                f"\n\n以下是前期调研发现的 Research Gaps，请优先围绕这些方向设计方案：\n"
                f"{json.dumps(gaps[:3], ensure_ascii=False)}\n"
            )

        system_prompt = (
            "你是材料科学专家。根据以下文献检索结果，设计候选材料合成方案。\n"
            f"参数及合法取值：{json.dumps(keys, ensure_ascii=False)}\n"
            "每个参数的值必须从对应合法取值中选取。\n"
            "严格返回JSON数组，不要附加任何文字。"
            f"{gaps_instruction}"
        )
        user_prompt = f"文献检索结果：\n{literature_text}\n\n请生成{POPULATION_SIZE}个候选方案。"

        ga_log("调用LLM生成初始种群...")
        raw = self._call_llm(system_prompt, user_prompt)
        candidates = parse_llm_json(raw)

        self.population = []
        for c in candidates[:POPULATION_SIZE]:
            valid_genes = {}
            for k in keys:
                val = c.get(k)
                valid_genes[k] = val if val in SEARCH_SPACE[k] else random.choice(SEARCH_SPACE[k])
            self.population.append(Individual(valid_genes))

        while len(self.population) < POPULATION_SIZE:
            base = random.choice(self.population)
            self.population.append(self._mutate(copy.deepcopy(base)))

        ga_log(f"初始种群生成完毕: {len(self.population)} 个个体")

    # ---------- 2. 证据驱动适应度评估 [v5.1] ----------
    def evaluate_fitness(self):
        # 准备文献证据文本
        evidence_text = json.dumps(getattr(self, 'literature_evidence', []), ensure_ascii=False, indent=2)
        if not evidence_text.strip() or evidence_text == "[]":
            evidence_text = "（暂无可用的文献检索结果，请基于领域通识评估）"

        system_prompt = (
            "你是严格的科学证据评审专家。请基于提供的文献检索结果，对给定材料/工艺方案进行证据驱动的多维评估。\n\n"
            "评估维度（每项 0-1）：\n"
            "1. evidence_score：是否有具体文献证据支持该方案？有 DOI 支持的直接证据得分高，无证据或推断得分低。\n"
            "2. mechanism_score：结构-性能关系是否有合理的物理/化学/学科机制解释？因果链越完整得分越高。\n"
            "3. relevance_score：该方案是否直接回答用户的研究问题？\n"
            "4. novelty_score：该组合/关系是否已被明确报道？组合越新得分越高，但不得为编造证据服务。\n"
            "5. overall_score：综合得分 = 0.4*evidence_score + 0.3*mechanism_score + 0.2*relevance_score + 0.1*novelty_score。\n\n"
            "输出格式（严格 JSON，不要附加任何解释文字）：\n"
            "{\n"
            '  "evidence_score": 0.75,\n'
            '  "mechanism_score": 0.80,\n'
            '  "relevance_score": 0.90,\n'
            '  "novelty_score": 0.40,\n'
            '  "overall_score": 0.72,\n'
            '  "claims": [\n'
            '    {"claim": "具体断言", "supporting_doi": "DOI或null", "evidence_type": "direct/indirect/inferred/none", "notes": "说明"}\n'
            "  ],\n"
            '  "mechanism": "简要机制解释",\n'
            '  "uncertainty": "主要不确定性",\n'
            '  "novelty_class": "known | known_combination | novel_combination | inferred"\n'
            "}\n\n"
            "重要规则：\n"
            "- 没有 DOI 或明确文献来源的断言，evidence_type 必须为 none/inferred。\n"
            "- 禁止编造 DOI 或具体数值。\n"
            "- novelty_score 高不直接提升 overall_score；overall_score 由证据和机制主导。\n"
        )

        for i, ind in enumerate(self.population):
            user_prompt = (
                f"方案({i+1}/{len(self.population)}):\n{json.dumps(ind.genes, ensure_ascii=False, indent=2)}\n\n"
                f"用户研究问题/目标性质：{state.get('context', {}).get('target_property', '构效关系发现')}\n\n"
                f"文献检索结果摘要：\n{evidence_text[:2500]}"
            )
            ga_log(f"  评估个体 {i+1}/{len(self.population)}")
            try:
                raw = self._call_llm(system_prompt, user_prompt)
                result = parse_llm_json(raw)

                # 提取各维度分数
                evidence = float(result.get("evidence_score", 0))
                mechanism = float(result.get("mechanism_score", 0))
                relevance = float(result.get("relevance_score", 0))
                novelty = float(result.get("novelty_score", 0))
                overall = float(result.get("overall_score",
                    0.4*evidence + 0.3*mechanism + 0.2*relevance + 0.1*novelty))

                ind.fitness = max(0.0, min(10.0, overall * 10.0))

                # 构建结构化理由
                claims = result.get("claims", [])
                claims_str = " | ".join([
                    f"{c.get('claim', '')[:60]}({c.get('evidence_type', 'none')})"
                    for c in claims[:3]
                ])
                reason = (
                    f"overall={overall:.2f} evidence={evidence:.2f} mechanism={mechanism:.2f} "
                    f"relevance={relevance:.2f} novelty={novelty:.2f} | "
                    f"机制: {result.get('mechanism', '')[:80]} | "
                    f"断言: {claims_str}"
                )
                ind.eval_reason = reason
            except Exception as e:
                ga_log(f"    ⚠ 评估异常: {e}")
                ind.fitness = 0.0
                ind.eval_reason = f"评估失败: {e}"

    # ---------- 3-5. 选择/交叉/变异 ----------
    def _roulette_select(self) -> Individual:
        total = sum(i.fitness for i in self.population)
        if total == 0:
            return random.choice(self.population)
        r = random.uniform(0, total)
        cum = 0.0
        for ind in self.population:
            cum += ind.fitness
            if cum >= r:
                return ind
        return self.population[-1]

    def _crossover(self, p1: Individual, p2: Individual) -> tuple:
        if random.random() > CROSSOVER_RATE:
            return copy.deepcopy(p1), copy.deepcopy(p2)
        keys = list(SEARCH_SPACE.keys())
        pt = random.randint(1, len(keys) - 1)
        c1, c2 = {}, {}
        for idx, k in enumerate(keys):
            if idx < pt:
                c1[k], c2[k] = p1.genes[k], p2.genes[k]
            else:
                c1[k], c2[k] = p2.genes[k], p1.genes[k]
        return Individual(c1), Individual(c2)

    def _mutate(self, ind: Individual) -> Individual:
        for k in SEARCH_SPACE:
            if random.random() < MUTATION_RATE:
                ind.genes[k] = random.choice(SEARCH_SPACE[k])
        return ind

    # ---------- 6. 进化一代 ----------
    def evolve_one_generation(self):
        new_pop = []
        best = max(self.population, key=lambda x: x.fitness)
        elite = copy.deepcopy(best)
        elite.eval_reason = "[精英保留] " + elite.eval_reason
        new_pop.append(elite)

        while len(new_pop) < POPULATION_SIZE:
            c1, c2 = self._crossover(self._roulette_select(), self._roulette_select())
            new_pop.append(self._mutate(c1))
            if len(new_pop) < POPULATION_SIZE:
                new_pop.append(self._mutate(c2))
        self.population = new_pop

    # ---------- 7. 保存结果 ----------
    def save_best(self):
        best = max(self.population, key=lambda x: x.fitness)
        record = {
            "generation": self.generation,
            "best_individual": best.to_dict(),
            "avg_fitness": round(sum(i.fitness for i in self.population) / len(self.population), 3),
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
        }
        self.history.append(record)
        with open(RESULTS_FILE, "w", encoding="utf-8") as f:
            json.dump(self.history, f, ensure_ascii=False, indent=2)
        ga_log(f"💾 第{self.generation}代最优已保存 | 得分:{best.fitness:.2f} | {best.eval_reason[:80]}")

    # ---------- 主循环 ----------
    def run(self):
        ga_log("=" * 50)
        ga_log(f"🧬 GA搜索启动 | 代数:{MAX_GENERATIONS} | 种群:{POPULATION_SIZE}")
        ga_log("=" * 50)

        self.initialize_population()
        self.evaluate_fitness()
        self.save_best()

        for gen in range(1, MAX_GENERATIONS + 1):
            self.generation = gen
            ga_log(f"\n--- 第 {gen}/{MAX_GENERATIONS} 代 ---")
            self.evolve_one_generation()
            self.evaluate_fitness()
            self.save_best()

        final = max(self.population, key=lambda x: x.fitness)
        ga_log(f"\n✅ 搜索完成 | 🏆 最优: {final.genes} | 得分: {final.fitness:.2f}")

        # [迁移改动] GA 完成后更新 state
        state["phase"] = "discovery"
        state["ga_complete"] = True
        ga_log(f"📊 state 已更新: phase={state['phase']}, ga_complete={state['ga_complete']}")

# ══════════════════════════════════════════════════════════════

# 以下为 GA 测试代码（保留原版，用清晰注释标记）

# ══════════════════════════════════════════════════════════════

# ==================== 项目全局变量（安全声明）====================

# 仅在变量完全不存在时才设默认值，绝不覆盖已有值

_globals_defaults = {
    'SCIVERSE_API_TOKEN': "",
    'sciverse_client': None,
    'SCIVERSE_DATA_DIR': "",
    'CACHE_FILE_PATH': "",
    'LOG_FILE_PATH': "",
    '_api_connected': False,
}
for _var_name, _default_val in _globals_defaults.items():
    if _var_name not in globals():
        globals()[_var_name] = _default_val
del _globals_defaults, _var_name, _default_val

# [迁移改动] SCIVERSE_API_TOKEN 从 CONFIG 读取（兼容旧代码）

if not SCIVERSE_API_TOKEN:
    SCIVERSE_API_TOKEN = CONFIG["sciverse_token"]

def run_ga_phase():
    """
    [v4新增] 运行 GA 发现阶段。
    该函数不会自动执行，需由统一编排器调用。
    调用前请确保 state["context"]["gaps"] 已写入。
    """
    global sciverse_client

    if not _api_connected:
        print("⚠️ API未连接，跳过GA搜索")
        return None

    # 如果 client 为 None，重新创建
    if sciverse_client is None:
        print("⚠️ sciverse_client 为 None，正在重新创建...")
        sciverse_client = SciverseClient(api_token=CONFIG["sciverse_token"], state=state)
        print(f"✅ 重建成功: {type(sciverse_client)}")

    ga = MaterialGA()
    ga.run()

    # [迁移改动] state 更新已在 ga.run() 内部完成，这里做持久化
    state["ga_complete"] = True
    print("✅ GA 搜索完成")

    # 保存状态
    with open("agent_state.json", "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)
        print("💾 agent_state.json 已保存")

    return ga


print("✅ Cell 6 遗传算法模块加载完成")
print("   调用方式：run_ga_phase()")
print("   注意：本 Cell 不再自动执行 GA，请通过 Cell 8 的统一编排器启动。")


In [ ]:
# -*- coding: utf-8 -*-
# Cell 7: Agent主循环 (v5.1 兼容证据驱动评估工具)

# ══════ 🔒 线性防护：检查前置依赖 ══════
_required = ["CONFIG", "state", "TOOL_REGISTRY", "TOOLS", "SYSTEM_PROMPT",
"llm_client", "sciverse_client"]
_missing = [v for v in _required if v not in dir() and v not in globals()]
if _missing:
    raise RuntimeError(f"❌ 请先运行定义以下变量的 Cell: {_missing}")

# ══════ 🔒 线性防护 - 通过 ══════

# ============================================================
# [v3重构] Cell 7: run_agent - 双循环阶段化搜索状态机
# ============================================================

# 为什么重构：
# 原版 run_agent 使用"自由循环"模式：LLM 自己决定何时搜索、何时停止。
# 问题：(1) 搜索策略不可控，可能过早收敛或无限发散；
#       (2) 缺少证伪环节，结论可靠性不足；
#       (3) 无法保证搜索的广度和深度。
#
# 新方案："双循环阶段化搜索" - 6个阶段，代码控制阶段转换，
# 每个阶段有明确的搜索策略和轮次限制。
# 外循环（发散）：Phase 1 + Phase 4 - 广泛发现线索
# 内循环（收敛）：Phase 2 + Phase 5 - 聚焦验证假设
# 关键检查点：Phase 3（证伪）- 确保结论可靠性
# 最终出口：Phase 6（验证）- 输出结构化结果

import json
from openai import OpenAI

# ============================================================
# [v3新增] 阶段化搜索的 Prompt 模板
# ============================================================

# 每个阶段开始时注入到 messages 中的 system prompt，
# 引导 LLM 在当前阶段使用对应的搜索策略。
# 设计原则：明确告知 LLM 当前所处阶段、该做什么、不该做什么。

PHASE_PROMPTS = {
    "diverge_1": (
       "【阶段1/6：发散搜索】你的任务是广泛搜索相关文献，建立领域全景。\n"
        "本阶段采用'摘要优先'策略：所有搜索工具默认返回 title + abstract + DOI + year + venue。\n"
       "不要获取全文，先低成本获取大量题录+摘要进行筛选。\n"
        "请按以下顺序执行搜索：\n"
        "  1. 使用 search_literature 进行宽泛关键词搜索（page_size=10），覆盖主要方向\n"
        "  2. 使用 search_weak 搜索近义词、上位概念、跨领域关键词，补充盲区\n"
        "  3. 使用 search_related 从轮次1-2的结果中提取关键实体（论文名、作者名、算法名），继续追踪相关工作\n"
        "目标：建立对该研究领域的全面认识，识别潜在 Research Gap，不要过早聚焦。"
    ),
    "converge_1": (
        "【阶段2/6：收敛搜索】基于上一阶段的摘要和题录，聚焦最有可能的2-3个假设。\n"
        "继续使用 search_literature / search_weak / search_related 收集支持/反驳证据，\n"
        "重点关注：哪些假设有摘要级证据支持？证据缺口在哪里？\n"
        "本阶段仍不要获取全文，只通过题录和摘要进行筛选。"
    ),
    "falsify": (
        "【阶段3/6：证伪检查】现在请对最有证据支持的假设调用 falsification_check。\n"
        "将你的假设和目前收集到的证据摘要传入，让审稿人视角来检查漏洞。\n"
        "如果某篇文献对证伪至关重要且 abstract 明显不够用，可以调用 fetch_paper_content(level='abstract') 再确认一下。"
    ),
    "diverge_2": (
        "【阶段4/6：拓展发散】证伪检查通过后，继续围绕已确认的方向进行拓展搜索：\n"
        "  1. 使用 search_related 搜索相关算法/方法的变体（relation_type='similar'）\n"
        "  2. 使用 search_related 搜索关键作者的其他工作（relation_type='author'）\n"
        "  3. 使用 search_weak 搜索其他领域的类似问题解法，寻找跨领域灵感\n"
        "本阶段仍以摘要和题录为主，仅对极少数核心文献使用 fetch_paper_content。"
    ),
    "converge_2": (
        "【阶段5/6：收敛整合】整合所有搜索发现，构建完整的证据链。\n"
        "使用 search_literature 和 search_related 针对最终假设补充最后的证据。\n"
        "重点：搜索证据链中缺失的环节。\n"
        "对于证据链上的关键文献（最多2-3篇），可以调用 fetch_paper_content(level='abstract' 或 'fulltext') 获取更完整内容。"
    ),
    "final_verify": (
        "【阶段6/6：最终验证】所有搜索已完成。请基于全部搜索结果，输出最终的结构化 JSON 结果。\n"
        "注意：你当前能访问的文献数据包括标题、摘要、DOI、年份、期刊。\n"
        "如果某些文献没有拿到摘要或全文，请在输出中明确标注，不要编造。\n"
        "请严格使用以下 JSON 格式，不要包含任何解释文字或 markdown 代码块标记：\n"
        "{\n"
        "  \"research_gaps\": [\"gap1\", \"gap2\", ...],  // Research Gap 列表，每个条目为字符串\n"
        "  \"key_findings\": [\"发现1\", \"发现2\", ...],  // 关键文献发现\n"
        "  \"evidence_chain\": [\"证据1（含文献来源/DOI，注明是否来自摘要/全文）\", \"证据2\", ...],\n"
        "  \"hypotheses\": [\"假设1\", \"假设2\", ...],  // 基于证据的候选构效关系假设\n"
        "  \"future_directions\": [\"方向1\", \"方向2\"],  // 未来研究方向\n"
        "  \"confidence\": \"high|medium|low\",  // 对结论整体置信度\n"
        "  \"data_limitations\": \"说明哪些证据来自摘要，哪些来自全文，哪些缺失\"\n"
        "}\n"
        "注意：\n"
        "  - research_gaps 必须是字符串数组，这是下游 GA 发现阶段的输入。\n"
        "  - 每条 gap 请写明具体缺失的知识连接或待验证假设。\n"
        "  - 不要再调用搜索工具，直接输出最终结果。"
    ),
}

# [v3新增] 阶段执行顺序 - 代码按此顺序自动推进
# 这是"双循环"的核心：发散(diverge) → 收敛(converge) → 证伪(falsify) → 拓展发散 → 收敛整合 → 最终验证
PHASE_ORDER = [
    "diverge_1",     # 阶段1：发散搜索（外循环起点）
    "converge_1",    # 阶段2：收敛搜索（内循环起点）
    "falsify",       # 阶段3：证伪检查（关键检查点）
    "diverge_2",     # 阶段4：拓展发散（外循环第二轮）
    "converge_2",    # 阶段5：收敛整合（内循环第二轮）
    "final_verify",  # 阶段6：最终验证（出口）
]

def run_agent(user_message: str, max_tool_rounds: int = None):
    """
    [v3重构] 双循环阶段化搜索 Agent

    核心改进：将原来的"自由循环"改为"阶段状态机"。
    - 6个阶段，代码控制阶段转换（不是 LLM 自己决定）
    - 每个阶段有最大轮次限制（从 CONFIG["search_phases"] 读取）
    - 每个阶段开始时注入对应的 system prompt
    - 保留原有所有防护机制（查询去重、异常捕获、JSON解析重试等）

    参数:
    user_message: 用户的研究问题
    max_tool_rounds: 全局最大轮次（可选，默认从 CONFIG 自动计算）

    返回:
    解析后的 JSON 结果字典
    """
    # [v3新增] 自动计算全局最大轮次
    # 如果调用者没有指定 max_tool_rounds，则从 CONFIG 的 search_phases 求和
    if max_tool_rounds is None:
        phases_config = CONFIG.get("search_phases", {})
        max_tool_rounds = sum(phases_config.values())
        # 加 2 轮余量，用于 JSON 解析重试等意外情况
        max_tool_rounds += 2

    # ========== 初始化 messages ==========
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message}
    ]

    # ========== 防护机制初始化（保留原版所有防护） ==========
    executed_queries: set = set()          # 防护1: 查询去重
    consecutive_empty = 0                   # 防护5: 连续空内容计数
    MAX_CONSECUTIVE_EMPTY = 2              # 防护5: 连续空内容上限

    # ========== [v3新增] 阶段状态机变量 ==========
    current_phase_idx = 0                   # 当前阶段在 PHASE_ORDER 中的索引
    current_phase_name = PHASE_ORDER[0]     # 当前阶段名称
    phase_round_count = 0                   # 当前阶段已消耗的轮次数
    total_round = 0                         # 全局轮次计数

    # 从 CONFIG 读取各阶段最大轮次
    phases_config = CONFIG.get("search_phases", {})

    # [v3新增] 注入第一个阶段的 system prompt
    # 这样 LLM 从一开始就知道自己处于哪个阶段、该做什么
    _first_phase_prompt = PHASE_PROMPTS.get(current_phase_name, "")
    if _first_phase_prompt:
        messages.append({"role": "system", "content": _first_phase_prompt})
        print(f"📍 进入阶段: {current_phase_name} (最大 {phases_config.get(current_phase_name, '?')} 轮)")

    # ========== 主循环 ==========
    for round_num in range(max_tool_rounds):
        # ---------- [v3新增] 阶段转换检查 ----------
        # 在每个 LLM 调用前，检查当前阶段是否已用完轮次
        # 如果是，自动推进到下一阶段
        _max_rounds_for_phase = phases_config.get(current_phase_name, 1)
        if phase_round_count >= _max_rounds_for_phase and current_phase_idx < len(PHASE_ORDER) - 1:
            # 当前阶段轮次已用完，推进到下一阶段
            current_phase_idx += 1
            current_phase_name = PHASE_ORDER[current_phase_idx]
            phase_round_count = 0

            # 注入新阶段的 system prompt
            _phase_prompt = PHASE_PROMPTS.get(current_phase_name, "")
            if _phase_prompt:
                messages.append({"role": "system", "content": _phase_prompt})
                print(f"📍 进入阶段: {current_phase_name} (最大 {phases_config.get(current_phase_name, '?')} 轮)")

                # 如果是最终验证阶段，提示 LLM 不要再调用工具
                if current_phase_name == "final_verify":
                    messages.append({
                        "role": "user",
                        "content": "你已进入最终验证阶段。请不要再调用任何搜索工具，"
                                   "直接基于之前所有搜索结果，输出最终的结构化 JSON 结果。"
                    })

        # ---------- API 调用 ----------
        try:
            response = llm_client.chat.completions.create(
                # [迁移改动] 模型名从 CONFIG 读取
                model=CONFIG["llm_model"],
                messages=messages,
                tools=TOOLS,
                temperature=0.5,
                extra_body={"thinking": {"type": "disabled"}},
            )
        except Exception as e:
            print(f"❌ [Round {round_num}] API错误: {type(e).__name__}: {e}")
            raise

        msg = response.choices[0].message
        finish_reason = response.choices[0].finish_reason
        print(f"[Round {round_num}] ✅ finish_reason={finish_reason}, "
              f"tool_calls={len(msg.tool_calls) if msg.tool_calls else 0} | "
              f"阶段={current_phase_name} ({phase_round_count}/{phases_config.get(current_phase_name, '?')})")

        # ========== 分支A: 模型请求调用工具 ==========
        if msg.tool_calls:
            messages.append(msg)

            for tc in msg.tool_calls:
                func_name = tc.function.name
                try:
                    func_args = json.loads(tc.function.arguments)
                except json.JSONDecodeError:
                    func_args = {}

                # --- 防护1: 查询去重 ---
                query_key = f"{func_name}:{json.dumps(func_args, sort_keys=True)}"
                if query_key in executed_queries:
                    tool_result = {
                        "warning": "此查询已在之前的轮次中执行过且结果相同。"
                                   "请基于已有搜索结果进行分析并直接输出最终JSON，不要重复搜索。"
                    }
                    print(f"  ⚠️ 重复查询已拦截: {func_name}({func_args})")
                else:
                    executed_queries.add(query_key)
                    # --- 防护2: 工具执行异常捕获 ---
                    try:
                        if func_name in TOOL_REGISTRY:
                            tool_result = TOOL_REGISTRY[func_name]["func"](**func_args)
                        else:
                            tool_result = {"error": f"未注册的工具: {func_name}"}
                        print(f"  🔧 {func_name}({func_args}) → 成功")
                    except Exception as tool_err:
                        tool_result = {"error": f"{type(tool_err).__name__}: {tool_err}"}
                        print(f"  ❌ {func_name}({func_args}) → 失败: {tool_err}")

                # --- 防护3: 工具结果不能为空字符串 ---
                result_str = json.dumps(tool_result, ensure_ascii=False)
                if not result_str or result_str == '""':
                    result_str = json.dumps({"info": "工具返回了空结果"}, ensure_ascii=False)

                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result_str
                })

            # [v3改动] 工具调用后，当前阶段的轮次计数 +1
            phase_round_count += 1
            total_round += 1
            consecutive_empty = 0
            continue

        # ========== 分支B: 模型返回文本（尝试解析JSON） ==========
        content = (msg.content or "").strip()

        # --- 防护4: 空内容处理 ---
        if not content:
            consecutive_empty += 1
            print(f"  ⚠️ 空内容 (连续第{consecutive_empty}次)")
            messages.append({"role": "assistant", "content": ""})
            if consecutive_empty >= MAX_CONSECUTIVE_EMPTY:
                messages.append({
                    "role": "user",
                    "content": "你已连续多次返回空内容。请立即基于已有信息输出最终JSON结果，不要再调用工具。"
                })
            else:
                messages.append({
                    "role": "user",
                    "content": "你的回复为空。请直接输出JSON格式的结果。"
                })
            continue

        consecutive_empty = 0

        # --- 去除 markdown 代码块包裹 ---
        if content.startswith("```"):
            lines = content.split("\n")
            start = 1
            end = len(lines)
            if lines[-1].strip() == "```":
                end = -1
            content = "\n".join(lines[start:end]).strip()

        # --- 防护5: JSON 解析 + 阶段守卫 ---
        # [v5修复] 非 final_verify 阶段不允许直接输出 JSON 结束
        # 防止 LLM "偷懒" 跳过搜索直接给结果
        if current_phase_name != "final_verify":
            print(f"  ⚠️ 阶段守卫: {current_phase_name} 阶段不允许直接输出结果，强制继续搜索")
            messages.append({"role": "assistant", "content": content})
            messages.append({
                "role": "user",
                "content": (
                    f"你当前处于【{current_phase_name}】阶段，尚未完成本阶段的搜索任务。"
                    "请不要直接输出最终结果。"
                    "请继续使用搜索工具（search_literature / search_weak / search_related / fetch_paper_content / verify_doi_reference / extract_claims / check_mechanism_consistency）"
                    "完成本阶段的搜索目标，只有在 final_verify 阶段才能输出最终 JSON。"
                )
            })
            phase_round_count += 1
            continue

        # 只有 final_verify 阶段才允许解析 JSON 并结束
        try:
            parsed = json.loads(content)
            print(f"[Round {round_num}] 🎉 JSON解析成功，Agent完成")
            print(f"📊 搜索统计: 总计 {total_round} 轮工具调用, {len(executed_queries)} 次不重复查询")
            return parsed
        except json.JSONDecodeError as e:
            print(f"  ⚠️ JSON解析失败: {e}")
            print(f"  📄 原始内容前200字: {content[:200]}")
            messages.append({"role": "assistant", "content": content})
            messages.append({
                "role": "user",
                "content": (
                    f"你上次输出的JSON无效（错误: {e}）。"
                    "请仅输出合法的JSON对象，不要包含任何解释文字、markdown标记或代码块符号。"
                )
            })
            phase_round_count += 1
            continue

print("✅ run_agent (v3 双循环阶段化版本) 定义完成")


In [ ]:
# -*- coding: utf-8 -*-
# Cell 8: 统一编排入口（v5.1 证据驱动 pipeline）
# 本 Cell 负责把文献调研阶段（Cell 7）和材料发现阶段（Cell 6）串联起来

import json
import re

def extract_gaps_from_research_result(result: dict) -> list:
    """
    从 run_agent 返回的 JSON 中提取 Research Gaps。
    兼容多种可能的字段名和嵌套格式。
    """
    if not isinstance(result, dict):
        return []

    # 优先读取标准字段
    for key in ["research_gaps", "gaps", "research_gaps_list", "identified_gaps"]:
        gaps = result.get(key)
        if isinstance(gaps, list):
            return [str(g).strip() for g in gaps if g]

    # 兜底：从文本字段中按行提取包含 "gap" 或 "不足" / "缺失" 的句子
    for key in ["summary", "conclusion", "findings", "output", "analysis"]:
        text = result.get(key, "")
        if not isinstance(text, str):
            continue
        candidates = re.split(r'[。\n]', text)
        gaps = [c.strip() for c in candidates
                if any(kw in c.lower() for kw in ["gap", "不足", "缺失", "空白", "未解决", "不明确"])]
        if gaps:
            return gaps

    return []


def run_discovery_pipeline(user_message: str):
    """
    [v5.1] 统一编排器：文献调研 → 提取 Gaps → GA 材料发现 → 整合报告
    """
    # ---------- Phase 1: 文献调研 ----------
    print("=" * 60)
    print("📚 Phase 1: 文献调研")
    print("=" * 60)
    research_result = run_agent(user_message)

    # ---------- 桥接：提取 Research Gaps 并写入 state ----------
    gaps = extract_gaps_from_research_result(research_result)
    state["context"]["gaps"] = gaps
    state["phase"] = "discovery"

    print("\n" + "=" * 60)
    print(f"📊 从文献调研结果中提取到 {len(gaps)} 个 Research Gap")
    print("=" * 60)
    for i, gap in enumerate(gaps, 1):
        print(f"  {i}. {gap[:120]}{'...' if len(gap) > 120 else ''}")

    if not gaps:
        print("  ⚠️ 未提取到 Research Gap，GA 将使用默认 query 运行")

    # 持久化状态
    with open("agent_state.json", "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)

    # ---------- Phase 2: GA 材料发现 ----------
    print("\n" + "=" * 60)
    print("🧬 Phase 2: 遗传算法材料发现")
    print("=" * 60)
    ga = run_ga_phase()

    # ---------- Phase 3: 整合最终报告 ----------
    print("\n" + "=" * 60)
    print("📝 Phase 3: 结果整合")
    print("=" * 60)

    best_candidate = None
    if ga and ga.history:
        best_candidate = ga.history[-1].get("best_individual")

    final_report = {
        "research_gaps": gaps,
        "literature_result": research_result,
        "ga_history": ga.history if ga else [],
        "best_candidate": best_candidate,
        "total_generations": ga.generation if ga else 0,
        "phase": state.get("phase"),
        "ga_complete": state.get("ga_complete"),
    }

    return final_report


# 测试入口
TEST_TOPIC = (
    '针对锂离子电池正极材料 LiFePO₄，建立掺杂元素 + 掺杂比例 → 倍率性能"的构效关系。'
    '要求输出形式为 y = f(x)，其中 y 必须包含具体单位和数值范围'
    '（例如：0.1C 下可逆比容量 mAh/g、10C 下容量保持率 %、锂离子扩散系数 cm²/s），'
    'x 必须包含掺杂位点（Fe 位 / Li 位）、晶格畸变程度、掺杂后晶胞体积变化。'
    '请给出每个预测值的置信区间或误差范围，并说明模型来源。'
)

RUN_DEMO = False
print("\n" + "=" * 60)
print("📄 最终报告")
print("=" * 60)
if RUN_DEMO:
    result = run_discovery_pipeline(TEST_TOPIC)
    print(json.dumps(result, indent=2, ensure_ascii=False))


In [ ]:
test_search = search_literature(
    query="LiFePO4 magnesium doping rate performance",
    page_size=2,
)

print(json.dumps(test_search, ensure_ascii=False, indent=2))

## ✅ 测试问题：设置完整问题、简短检索词和年份

`SMALL_TOPIC` 保留完整研究问题；`query` 只填写适合学术数据库的简短专业关键词；`year_min` 是最早发表年份。以后更换课题时同时修改这三项。

In [ ]:
# ============================================================
# ✅ 测试问题：完整问题、简短检索词、最早发表年份
# ============================================================
SMALL_TOPIC = """
请检索近10年关于Mg掺杂LiFePO4正极材料的实验研究。

重点分析以下假设：
少量Mg优先取代LiFePO4中的Fe位，并通过改善电子电导率或锂离子扩散性能，
提高材料在5C至10C倍率下的比容量和容量保持率。

请重点检索：
1. Mg的掺杂位置；
2. Mg掺杂比例；
3. 烧结温度；
4. 0.1C、5C和10C下的比容量；
5. 锂离子扩散系数或电荷转移电阻；
6. 支持或反驳上述假设的实验结果。

筛选不超过8篇最相关的论文，列出Research Gaps。
所有数值和结论必须标注文献来源；证据不足时明确说明。
"""

query = "magnesium Mg-doped LiFePO4 cathode experimental 5C 10C"
year_min = 2016

print("✅ SMALL_TOPIC 已设置为Mg掺杂LiFePO4测试问题")
print(SMALL_TOPIC)
print(f"✅ 简短检索词：{query}")
print(f"✅ 最早发表年份：{year_min}")

## LangGraph 迭代深化循环

下面两个单元格将现有模块接入 `StateGraph`：

1. 构造与原代码字段一致的 `ResearchState`；
2. 运行 `search_node → analyze_node → gen_keywords_node → search_node` 循环；
3. 达到轮次上限、评分阈值、无新 Gap 或 Gap 重复时结束。

In [ ]:
# Cell 9: LangGraph StateGraph 初始化（可独立运行）
import json

from config import CONFIG as MODULE_CONFIG
from research_state_graph import ResearchState, graph

if "CONFIG" not in globals():
    CONFIG = MODULE_CONFIG

if "SMALL_TOPIC" not in globals():
    SMALL_TOPIC = """
请检索近15年关于“金刚石表面金属化或引入活性元素，如何影响金刚石—铜界面结合与热传输性能”的实验研究。
重点关注 Ti、Cr、Mo、W、Zr 等能够形成碳化物的界面元素或金属层。筛选不超过8篇与问题最直接相关的论文。
请寻找相关文献，列出 Research Gaps，
所有结论必须标注文献来源；证据不足时明确说明。
"""
    print("⚠️ 未检测到 SMALL_TOPIC，已自动加载默认研究题目")

if "query" not in globals():
    query = next(line.strip() for line in SMALL_TOPIC.splitlines() if line.strip())
    print("⚠️ 未检测到简短检索词 query，暂时使用研究问题的第一行")

if "year_min" not in globals():
    year_min = None
    print("⚠️ 未检测到 year_min，本次检索不限制发表年份")

phases_config = CONFIG.get("search_phases", {})
max_tool_rounds = sum(phases_config.values()) + 2

initial_state: ResearchState = {
    "phase": "research",
    "context": {
        "gaps": [],
        "cached_literature": [],
        "target_property": query,
    },
    "api_stats": {
        "total_calls": 0,
        "cache_hits": 0,
        "errors": 0,
    },
    "query": query,
    "year_min": year_min,
    "page_size": 8,
    "status": "",
    "error": "",
    "api_error": None,
    "total_returned": 0,
    "papers": [],
    "raw": {},
    "research_result": {},
    "gaps": [],
    "keywords": [],
    "executed_queries": set(),
    "total_round": 0,
    "max_tool_rounds": max_tool_rounds,
    "hypothesis": SMALL_TOPIC,
    "evidence_summary": "",
    "domain": "materials_science",
    "claims": [],
    "vulnerabilities": [],
    "rebuttals": {},
    "required_experiments": "",
    "verdict": "未知",
    "score": 0.0,
    "is_consistent": False,
}

print("✅ ResearchState 初始化完成")
print(f"   query={initial_state['query']}")
print(f"   year_min={initial_state['year_min']}, max_tool_rounds={max_tool_rounds}, page_size={initial_state['page_size']}")

In [ ]:
# Cell 10: 运行 LangGraph 迭代深化循环
# 本单元格会正式调用 Sciverse 和 DeepSeek API。
research_result = graph.invoke(
    initial_state,
    {"recursion_limit": max_tool_rounds * 3 + 1},
)

result_summary = {
    "status": research_result.get("status"),
    "error": research_result.get("error", ""),
    "api_error": research_result.get("api_error"),
    "query": research_result.get("query"),
    "year_min": research_result.get("year_min"),
    "total_returned": research_result.get("total_returned"),
    "total_round": research_result.get("total_round"),
    "score": research_result.get("score"),
    "is_consistent": research_result.get("is_consistent"),
    "verdict": research_result.get("verdict"),
    "gaps": research_result.get("gaps", []),
    "keywords": research_result.get("keywords", []),
    "papers": [
        {
            "title": paper.get("title", ""),
            "doi": paper.get("doi", ""),
            "publication_published_year": paper.get("publication_published_year"),
        }
        for paper in research_result.get("papers", [])[:8]
    ],
    "research_result": research_result.get("research_result", {}),
}

print(json.dumps(result_summary, ensure_ascii=False, indent=2))